# IV-7 音源分離の評価と深層学習（ノートブック版）

教科書の IV-7 の 6 節のプログラムを、1 冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac（Apple シリコン、MPS）で全部で約 15 分（学習を試しの設定にしたとき）。Colab ではもっとかかることがあります）。

| 節 | 試すこと | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | 評価指標（SDR・SI-SDR）を自分で実装して照合し、落とし穴を数値で確かめる | B・C | `metrics_check.py` |
| 2 | 理想マスクの上限を ILRMA と並べ、聞く | D | `oracle_masks.py`（`masks.py`） |
| 3 | 学習の準備（部屋と音声の区間）、STFT と PIT の照合 | E | `sepdata.py`・`masknet.py` |
| 4 | 小さなマスク推定ネットワークの学習と評価（**ノートブックでは 300 回の試しの設定**） | F | `train_masknet.py`・`eval_masknet.py` |
| 5 | 統計で比べ、誤用を確かめる | G | `stats_iv7.py` |
| 6 | 確認問題の答え合わせ | H | `quiz_check.py` |

- 次のセルたちが、共通ライブラリ（IV-2 の `common/` の `__init__.py`・`stft.py`・`audio_io.py`・`plotting.py`、この章で足す `common/metrics.py`）と、第7章の `bss.py`、IV-6 の `bss6.py`、この章の部品 `masks.py`・`masknet.py` を、このノートブックの隣に `%%writefile` で書き出します。中身は教科書の各章の 6 節と同じなので、ほかのファイルは要りません。
- 各節のコードは 6 節のプログラムそのものです。変えたのは、ファイルの場所（`__file__` の代わりにノートブックのフォルダを使う。`sys.path.insert` も同じ）、図をセルの下に表示すること（`matplotlib.use("Agg")` を外し、`plt.show()` を足す）、コマンドの引数を `sys.argv` で渡すこと（同じプログラムを引数を変えて何度も実行するところは、`main()` をくり返し呼ぶ）、`masknet.py` の確かめを `masknet.self_check()` で呼ぶこと、出力をいったん `cap` に受け取って次のセルで表示し、`ok()` で教科書の手順の「確かめること」を確かめることだけです。
- **4 節（手順 F）の学習は重い**ので、ノートブックでは試しの設定（`--steps 300 --eval-every 100`：1 回 300 回、100 回ごとに検証と ckpt。執筆時の Mac で 1 回 30〜40 秒、9 回で約 6 分）を既定にしています。教科書の本番（2000 回、250 回ごと。1 回 5〜12 分、9 回で 1〜2 時間）にするときは、セルのコメントのとおりに `--steps` と `--eval-every` を消してください。試しの設定では、ネットワークの値（SDR 改善量・検証の SI-SDR・最良の回など）は教科書の数値と合いません。ノートブックの確かめは、学習の長さに関係なく成り立つことだけにしてあります（理想マスク・ILRMA・oracleW は学習を使わないので、教科書と同じ値になります）。
- **入れていないもの**：手順 A のうち `setup_iv7.sh`（ファイルを `~/home-lab/` に置く準備。ここでは `%%writefile` が代わり）と `run_iv7.sh`（全部を順に実行するシェルスクリプト。このノートブックが代わり）、手順 I（論文を読む。コードがない）。この章には録音の手順はありません。
- **音を聞くときの注意**：再生ボタンは `listen_file()` が作ります。保存した wav（最大 −10 dBFS）をそのままの大きさで鳴らすため、`IPython.display.Audio(..., normalize=False)` を使います。**`normalize=False` を消さないでください**（消すと最大値が 1 に引き上げられ、とても大きな音になります）。再生する前にパソコンの音量を一番小さくしてください。理想バイナリマスクの音にはピコピコ・チリチリとした音（ミュージカルノイズ）が入るので、長く聞き続けないでください。
- 話し声は librosa の例の音源 `libri1`・`libri2`・`libri3` を使います（初回はダウンロードします。ネットワークが要ります）。
- 数値は乱数の種を変えなければ教科書とほぼ同じですが、pyroomacoustics・scipy・fast_bss_eval の版で最後の桁が変わることがあります。`ok()` はその分の幅を持たせてあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

## 0. 準備（torch と計算の場所）

- **torch**：Google Colab には torch が最初から入っています（版は公式サイトで要確認（執筆時点の目安））。Mac では第0章で入れたものを使います（教科書は torch 2.14.1 で確かめた）。下のセルは、torch が無いときだけ `pip install torch` をします。
- **計算の場所（device）**：`masknet.pick_device()` が **cuda → mps → cpu** の順に自動で選びます（III-4 の `repro.py` と同じ考え方）。Mac（Apple シリコン）では `mps`、Colab で GPU のランタイムにしたときは `cuda`、GPU が無ければ `cpu` です。Colab で GPU を使うには、メニューの「ランタイム」→「ランタイムのタイプを変更」で GPU を選びます（項目の名前は公式ドキュメントで要確認）。CPU だけでも動きますが、学習は数倍遅くなります。
- 計算の場所によって、学習の結果の数値は少し変わります（MPS・CUDA の計算は決定的でないことがある。III-4）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・statsmodels・soundfile・librosa・torch が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("torch", "torch"), ("librosa", "librosa"), ("soundfile", "soundfile"), ("statsmodels", "statsmodels"),
                 ("pyroomacoustics", "pyroomacoustics"), ("fast_bss_eval", "fast_bss_eval"),
                 ("mir_eval", "mir_eval")]:          # mir_eval は照合だけに使う（任意。無ければ照合を省く）
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, torch, statsmodels, pyroomacoustics, librosa
print("numpy", numpy.__version__, "scipy", scipy.__version__, "torch", torch.__version__, "statsmodels", statsmodels.__version__,
      "pyroomacoustics", pyroomacoustics.__version__, "librosa", librosa.__version__)
print("CUDA:", torch.cuda.is_available(), " MPS:", torch.backends.mps.is_available())
print("（教科書は numpy 2.5.5・scipy 1.18.1・statsmodels 0.15.0・pyroomacoustics 0.10.1・fast_bss_eval 0.1.4・torch 2.14.1 で確かめた）")
import os
for d in ["common", "out", "runs"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次の 9 つのセルで、ほかの章のファイル（IV-2 の `common/`、第7章の `bss.py`、IV-6 の `bss6.py`）と、この章の `common/metrics.py`・`masks.py`・`masknet.py` を書き出します。

In [ ]:
%%writefile common/__init__.py
"""第IV部の共通ライブラリ（stft・audio_io・plotting。後の章で metrics などを足す）"""

In [ ]:
%%writefile common/stft.py
"""第IV部の共通ライブラリ：短時間フーリエ変換（STFT）と逆 STFT（IV-2 で定義）

約束（第IV部のすべての章で同じ）
  X = stft(x, L=1024, S=256, window="hann")
      x : 形 (n,) か (n, M)（M はマイクの数）
      X : 形 (F, T) か (F, T, M)、F = L//2 + 1、T = 1 + n//S（複素数、complex128）
  x = istft(X, L, S, window="hann", length=None)
      stft で作った X なら、元の x に戻る（完全再構成。誤差は 1e-12 程度）

決め方（librosa.stft の既定と同じ。第7章の bss.py・nmf.py の STFT と数値が一致する）
  ・窓は「周期的な」窓（scipy.signal.get_window の既定。hann なら w[n] = sin²(πn/L)）
  ・フレーム t の窓の中心が、信号の時刻 n = t·S に来る（前後に L//2 個のゼロを足す）
  ・位相の基準は窓の始まり（phase="start"）。phase="center" にすると窓の中心が基準になり、
    scipy.signal.ShortTimeFFT の既定（phase_shift=0）と同じになる
  ・正規化はしない（窓を掛けて FFT した値そのもの）
  ・逆 STFT の合成窓は、最小二乗の意味で最適な窓 g[n] = w[n] / Σ_t w²[n − tS]
    （Griffin & Lim 1984。端で重なりが少ないところも、実際に重なっているフレームだけで割る）
"""
import numpy as np
from scipy.signal import get_window as _scipy_get_window

__all__ = ["get_window", "stft", "istft", "window_sum", "dual_window",
           "bin_freqs", "frame_times", "n_frames"]


def get_window(window, L):
    """窓を長さ L の配列で返す。window は名前（"hann"、"hamming"、("kaiser", 8) など）か配列。
    名前のときは周期的な窓（fftbins=True）。"sqrthann" は hann の平方根（分析と合成に同じ窓を使うとき）。"""
    if isinstance(window, str) and window == "sqrthann":
        return np.sqrt(_scipy_get_window("hann", L, fftbins=True))
    if isinstance(window, (str, tuple)):
        return _scipy_get_window(window, L, fftbins=True).astype(float)
    w = np.asarray(window, dtype=float)
    if w.shape != (L,):
        raise ValueError(f"窓の長さ {w.shape} が L = {L} と合いません")
    return w


def n_frames(n, S):
    """長さ n の信号のフレームの数 T = 1 + n//S"""
    return 1 + n // S


def bin_freqs(L, fs):
    """周波数のビン k = 0, …, L//2 の周波数 [Hz]：k·fs/L"""
    return np.arange(L // 2 + 1) * fs / L


def frame_times(T, S, fs):
    """フレーム t = 0, …, T−1 の窓の中心の時刻 [s]：t·S/fs"""
    return np.arange(T) * S / fs


def _phase_factor(L, F, phase):
    """位相の基準を「窓の中心」にするときに掛ける因子 exp(+j2πk(L//2)/L)（L が偶数なら (−1)^k）"""
    if phase == "start":
        return None
    if phase == "center":
        k = np.arange(F)
        if L % 2 == 0:
            return (-1.0) ** k + 0j                      # L が偶数なら exp(jπk) = (−1)^k を正確に
        return np.exp(2j * np.pi * k * (L // 2) / L)
    raise ValueError('phase は "start" か "center"')


def stft(x, L=1024, S=256, window="hann", phase="start"):
    """短時間フーリエ変換。X[k, t] = Σ_{m=0}^{L−1} w[m] x[tS + m − L//2] e^{−j2πkm/L}（範囲外の x は 0）"""
    x = np.asarray(x, dtype=float)
    if x.ndim not in (1, 2):
        raise ValueError("x の形は (n,) か (n, M)")
    if not 0 < S <= L:
        raise ValueError("シフト S は 1 以上 L 以下")
    w = get_window(window, L)
    n = x.shape[0]
    T = n_frames(n, S)
    pad = L // 2
    # 前に L//2、後ろはフレームが全部入るだけのゼロを足す
    total = max((T - 1) * S + L, pad + n)
    xp = np.zeros((total,) + x.shape[1:])
    xp[pad:pad + n] = x
    idx = np.arange(L)[None, :] + S * np.arange(T)[:, None]          # (T, L) 各フレームの番号
    frames = xp[idx] * (w[None, :] if x.ndim == 1 else w[None, :, None])   # (T, L) か (T, L, M)
    X = np.fft.rfft(frames, axis=1)                                   # (T, F) か (T, F, M)
    X = np.moveaxis(X, 0, 1)                                          # (F, T) か (F, T, M)
    c = _phase_factor(L, X.shape[0], phase)
    if c is not None:
        X = X * (c[:, None] if X.ndim == 2 else c[:, None, None])
    return X


def window_sum(w, S, T):
    """T 個のフレームを S ずつずらして重ねたときの Σ_t w²[n − tS]（長さ (T−1)S + L）"""
    L = len(w)
    out = np.zeros((T - 1) * S + L)
    for t in range(T):
        out[t * S:t * S + L] += w ** 2
    return out


def dual_window(w, S):
    """無限に続くフレームを考えたときの最適な合成窓（双対窓）g[n] = w[n] / Σ_k w²[n + kS]
    （scipy.signal.ShortTimeFFT の dual_win と同じもの）"""
    L = len(w)
    acc = np.zeros(L)
    for k in range(-(L // S) - 1, L // S + 2):
        lo, hi = max(0, k * S), min(L, L + k * S)
        if lo < hi:
            acc[lo:hi] += w[lo - k * S:hi - k * S] ** 2
    if np.any(acc <= 0):
        raise ValueError("NOLA の条件が成り立たない（Σ w² が 0 の点がある）")
    return w / acc


def istft(X, L, S, window="hann", length=None, phase="start"):
    """逆 STFT（重み付き重ね合わせ加算）。
    y[n] = Σ_t w[n − tS]·ifft(X[:, t])[n − tS] / Σ_t w²[n − tS]
    X が本物の STFT なら完全再構成、加工した X なら「STFT が X に最も近い信号」（最小二乗）を返す。"""
    X = np.asarray(X)
    if X.ndim not in (2, 3):
        raise ValueError("X の形は (F, T) か (F, T, M)")
    F, T = X.shape[:2]
    if F != L // 2 + 1:
        raise ValueError(f"X の周波数の数 {F} が L//2+1 = {L // 2 + 1} と合いません")
    w = get_window(window, L)
    c = _phase_factor(L, F, phase)
    if c is not None:
        X = X * (np.conj(c)[:, None] if X.ndim == 2 else np.conj(c)[:, None, None])
    frames = np.fft.irfft(X, n=L, axis=0)                            # (L, T) か (L, T, M)
    total = (T - 1) * S + L
    y = np.zeros((total,) + X.shape[2:])
    wb = w if X.ndim == 2 else w[:, None]
    for t in range(T):
        y[t * S:t * S + L] += wb * frames[:, t]
    wss = window_sum(w, S, T)
    pad = L // 2
    if length is None:
        length = (T - 1) * S
    end = min(pad + length, total)
    seg = wss[pad:end]
    if np.any(seg < 1e-10 * wss.max()):
        raise ValueError("NOLA の条件が成り立たない：窓の2乗の和が 0 の時刻があり、そこは戻せない")
    out = np.zeros((length,) + X.shape[2:])
    out[:end - pad] = y[pad:end] / (seg if X.ndim == 2 else seg[:, None])
    return out

In [ ]:
%%writefile common/audio_io.py
"""第IV部の共通ライブラリ：音の読み書きと再生（IV-2 で定義）

  x, fs = load(path, fs=None)      # x は (n,)（モノラル）か (n, M)。float64、だいたい −1〜1
  save(path, x, fs)                # 16 ビットの wav で保存（はみ出す値は止めて知らせる）
  play(x, fs, gain_db=-20)         # 最大値を gain_db [dBFS] にそろえてから鳴らす（最初は小さい音で）
  x = record(sec, fs=16000)        # Mac の内蔵マイクで録音する（モノラル）

再生は sounddevice があればそれを使い、無ければ macOS の afplay で一時ファイルを鳴らす。
"""
import os
import pathlib
import shutil
import subprocess
import tempfile

import numpy as np
import soundfile as sf

__all__ = ["load", "save", "play", "record", "peak_dbfs"]


def load(path, fs=None):
    """wav・flac・ogg などを読む。fs を指定すると、その標本化周波数に変換する（librosa を使う）"""
    x, fs0 = sf.read(str(path), dtype="float64", always_2d=False)
    if fs is not None and fs != fs0:
        import librosa
        x = librosa.resample(x.T if x.ndim == 2 else x, orig_sr=fs0, target_sr=fs).T
        fs0 = fs
    return x, fs0


def peak_dbfs(x):
    """最大値の大きさ [dBFS]（1.0 が 0 dBFS）"""
    return 20 * np.log10(np.max(np.abs(x)) + 1e-12)


def save(path, x, fs):
    """16 ビット PCM の wav で保存する。|x| が 1 を超えるときは保存しない（音割れを防ぐ）"""
    x = np.asarray(x, dtype=float)
    if np.max(np.abs(x)) > 1.0:
        raise ValueError(f"最大値が {np.max(np.abs(x)):.3f} で 1 を超えています。小さくしてから保存してください"
                         "（例：x / np.max(np.abs(x)) * 0.5）")
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x, int(fs), subtype="PCM_16")
    return path


def play(x, fs, gain_db=-20.0, blocking=True):
    """最大値を gain_db [dBFS] にそろえて鳴らす（既定 −20 dBFS ＝ 最大値 0.1）。
    はじめは Mac の音量を小さくし、少しずつ上げる。gain_db は 0 より大きくできない。"""
    if gain_db > 0:
        raise ValueError("gain_db は 0 以下にしてください（0 dBFS より大きくすると音が割れます）")
    x = np.asarray(x, dtype=float)
    peak = np.max(np.abs(x))
    if peak == 0:
        print("無音なので鳴らしません")
        return
    y = x / peak * 10 ** (gain_db / 20)
    try:
        import sounddevice as sd
    except ImportError:
        sd = None
    if sd is not None:
        sd.play(y.astype(np.float32), int(fs))
        if blocking:
            sd.wait()
        return
    if shutil.which("afplay") is None:
        raise RuntimeError("sounddevice も afplay も見つかりません。save() で保存して、ほかのアプリで聞いてください")
    fd, tmp = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    try:
        sf.write(tmp, y, int(fs), subtype="PCM_16")
        subprocess.run(["afplay", tmp], check=True)
    finally:
        os.remove(tmp)


def record(sec, fs=16000):
    """内蔵マイクで sec 秒録音し、(n,) の配列で返す（sounddevice が必要）。
    はじめて使うと macOS がマイクの許可を聞いてくるので許可する（項目名は macOS の版で要確認）"""
    import sounddevice as sd
    x = sd.rec(int(sec * fs), samplerate=int(fs), channels=1, dtype="float32")
    sd.wait()
    x = x[:, 0].astype(float)
    print(f"録音しました：最大 {peak_dbfs(x):.1f} dBFS")
    return x

In [ ]:
%%writefile common/plotting.py
"""第IV部の共通ライブラリ：図の設定とスペクトログラムの表示（IV-2 で定義）

  import matplotlib; matplotlib.use("Agg")      # 図をファイルに保存するだけのとき（第7章と同じ）
  from common.plotting import specshow
  specshow(X, fs, S, ax=ax, title="…")          # X は stft の結果（複素数）か、振幅・パワー

日本語のフォントは既存の章と同じ（Hiragino Sans）。
"""
import numpy as np
import matplotlib

matplotlib.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

__all__ = ["specshow", "to_db"]


def to_db(X, power=False, ref=None, floor_db=-120.0):
    """X（複素数・振幅・パワー）を dB にする。振幅は 20log10、パワーは 10log10。
    ref を与えないと、最大値を 0 dB にする"""
    A = np.abs(X)
    if power:
        D = 10 * np.log10(A + 1e-20)
    else:
        D = 20 * np.log10(A + 1e-20)
    D -= (D.max() if ref is None else (10 if power else 20) * np.log10(ref))
    return np.maximum(D, floor_db)


def specshow(X, fs, S, ax=None, power=False, db_range=80.0, fmax=None, title=None, cmap="magma",
             colorbar=True):
    """スペクトログラムを表示する。X は (F, T)。横軸はフレームの窓の中心の時刻 t·S/fs [s]、
    縦軸はビンの周波数 k·fs/L [Hz]（L = 2(F−1)）。色は最大値から db_range [dB] 下までを表示する。"""
    import matplotlib.pyplot as plt
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError("specshow に渡す X は (F, T)。多チャネルなら X[:, :, m] のように1つ選ぶ")
    F, T = X.shape
    L = 2 * (F - 1)
    D = to_db(X, power=power)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    df, dt = fs / L, S / fs
    # 各ビン・各フレームの値を、その中心の周りの長方形で塗る
    extent = (-dt / 2, (T - 1) * dt + dt / 2, -df / 2, (F - 1) * df + df / 2)
    im = ax.imshow(D, origin="lower", aspect="auto", extent=extent, cmap=cmap,
                   vmin=-db_range, vmax=0, interpolation="nearest")
    ax.set_xlabel("時間 [s]")
    ax.set_ylabel("周波数 [Hz]")
    if fmax is not None:
        ax.set_ylim(0, fmax)
    if title:
        ax.set_title(title)
    if colorbar:
        ax.figure.colorbar(im, ax=ax, label="dB（最大を 0）")
    return im

In [ ]:
%%writefile common/metrics.py
# 第IV部の共通ライブラリ（IV-7 で追加）
"""第IV部の共通ライブラリ：音源分離の評価指標（IV-7 で定義）

  sdr(ref, est, filter_length=512)           BSS Eval の SDR（NumPy で自前。fast_bss_eval.sdr と一致）
  si_sdr(ref, est, zero_mean=False)          SI-SDR（Le Roux ら 2019。fast_bss_eval.si_sdr と一致。fast_bss_eval 0.1.4 は
                                             torch の無い環境で si_sdr が AttributeError になるので、そのときは
                                             fast_bss_eval.sdr(ref, est, filter_length=1) と比べる。metrics_check.py の [2]）
  snr(ref, est)                              尺度も含めてそのまま比べる SNR（参考。尺度で値が変わる）
  bss_decomp(refs, est, j, filter_length)    推定 est を s_target・e_interf・e_artif に分ける（Vincent ら 2006）
  bss_eval_np(refs, ests, filter_length)     SDR・SIR・SAR（自前。順番は与えられたまま）
  bss_eval(ref, est, filter_length=512, compute_permutation=True)
                                             fast_bss_eval.bss_eval_sources の薄い包み → (sdr, sir, sar, perm)
配列の形：ref・est は (n,) か (N, n)（N は音源の数、n は標本の数）。(N, n) なら行ごとに計算して (N,) を返す。
約束：BSS Eval の「許す歪み」は、正解を 0〜filter_length−1 標本遅らせたものの重ね合わせ（長さ filter_length の
FIR フィルタ）。信号の外側は 0 とみなす（mir_eval・fast_bss_eval と同じ）。値は dB。
"""
import numpy as np

__all__ = ["sdr", "si_sdr", "snr", "bss_decomp", "bss_eval_np", "bss_eval", "xcorr"]


def _rows(ref, est):
    ref = np.asarray(ref, dtype=float)
    est = np.asarray(est, dtype=float)
    if ref.shape != est.shape:
        raise ValueError(f"ref {ref.shape} と est {est.shape} の形が違います（長さをそろえてください）")
    return ref.reshape(-1, ref.shape[-1]), est.reshape(-1, est.shape[-1]), ref.ndim == 1


def xcorr(a, b, lags):
    """c[k] = Σ_n a[n]·b[n + k]（k = 0, …, lags−1）。信号の外側は 0。FFT で計算する"""
    nfft = 1 << int(np.ceil(np.log2(len(a) + len(b) + lags)))
    A = np.fft.rfft(a, nfft)
    B = np.fft.rfft(b, nfft)
    return np.fft.irfft(np.conj(A) * B, nfft)[:lags]


def _db(num, den):
    with np.errstate(divide="ignore", invalid="ignore"):
        return 10 * np.log10(num / den)


def sdr(ref, est, filter_length=512):
    """BSS Eval の SDR = 10 log10(‖s_target‖² / ‖est − s_target‖²)。
    s_target は、正解 s を 0〜L−1 標本遅らせた L 本の信号の張る空間への est の直交射影（L = filter_length）。
    正規方程式 R h = r（R は s の自己相関のテプリッツ行列、r は s と est の相互相関）を解くと
    ‖s_target‖² = rᵀh、‖est − s_target‖² = ‖est‖² − rᵀh（直交するので）"""
    from scipy.linalg import toeplitz
    R_, E_, one = _rows(ref, est)
    out = []
    for s, e in zip(R_, E_):
        acf = xcorr(s, s, filter_length)
        r = xcorr(s, e, filter_length)
        h = np.linalg.solve(toeplitz(acf), r)
        tgt = float(r @ h)
        out.append(_db(tgt, float(e @ e) - tgt))
    out = np.array(out)
    return out[0] if one else out


def si_sdr(ref, est, zero_mean=False):
    """SI-SDR = 10 log10(‖α s‖² / ‖α s − est‖²)、α = ⟨est, s⟩ / ‖s‖²（est に最も近い s の定数倍）。
    ρ = ⟨est, s⟩/(‖s‖‖est‖)（正規化した相関）を使うと SI-SDR = 10 log10(ρ² / (1 − ρ²))。
    zero_mean=True で先に両方の平均を引く（Conv-TasNet などの実装の慣習。Le Roux ら 2019 の定義には無い。
    fast_bss_eval の既定も False）"""
    R_, E_, one = _rows(ref, est)
    if zero_mean:
        R_ = R_ - R_.mean(axis=1, keepdims=True)
        E_ = E_ - E_.mean(axis=1, keepdims=True)
    alpha = np.sum(E_ * R_, axis=1) / np.sum(R_ ** 2, axis=1)
    tgt = alpha[:, None] * R_
    out = _db(np.sum(tgt ** 2, axis=1), np.sum((tgt - E_) ** 2, axis=1))
    return out[0] if one else out


def snr(ref, est):
    """SNR = 10 log10(‖s‖² / ‖s − est‖²)。尺度を合わせないので、est を 0.5 倍しただけで値が下がる"""
    R_, E_, one = _rows(ref, est)
    out = _db(np.sum(R_ ** 2, axis=1), np.sum((R_ - E_) ** 2, axis=1))
    return out[0] if one else out


def _project(refs, est, L):
    """est を「refs の全部の行を 0〜L−1 遅らせた N·L 本の信号」の空間へ射影し、
    (係数 c (N, L), 射影した信号（長さ n + L − 1）) を返す"""
    from scipy.signal import fftconvolve
    N, n = refs.shape
    nfft = 1 << int(np.ceil(np.log2(2 * n + L)))
    S = np.fft.rfft(refs, nfft, axis=1)
    G = np.zeros((N * L, N * L))
    for i in range(N):
        for j in range(N):
            # c_ij[d] = Σ_m s_i[m] s_j[m + d]（d = −(L−1)…L−1）。G の (i,k),(j,l) 成分は c_ij[k − l]
            c = np.fft.irfft(np.conj(S[i]) * S[j], nfft)
            d = np.subtract.outer(np.arange(L), np.arange(L))          # k − l
            G[i * L:(i + 1) * L, j * L:(j + 1) * L] = c[d % nfft]
    D = np.concatenate([xcorr(refs[i], est, L) for i in range(N)])
    coef = np.linalg.solve(G, D).reshape(N, L)
    proj = sum(fftconvolve(refs[i], coef[i]) for i in range(N))      # 長さ n + L − 1
    return coef, proj


def bss_decomp(refs, est, j, filter_length=512):
    """推定 est (n,) を、正解 refs (N, n) の j 番目に対して分解する（Vincent ら 2006）。
    s_target = P_{s_j} est、e_interf = P_{全部} est − s_target、e_artif = est − P_{全部} est
    （どれも長さ n + L − 1。est の後ろに 0 を足した信号との比較）"""
    refs = np.asarray(refs, dtype=float)
    est = np.asarray(est, dtype=float)
    L = filter_length
    _, s_target = _project(refs[j:j + 1], est, L)
    _, p_all = _project(refs, est, L)
    e_pad = np.concatenate([est, np.zeros(L - 1)])
    return s_target, p_all - s_target, e_pad - p_all


def bss_eval_np(refs, ests, filter_length=512):
    """SDR・SIR・SAR（自前）。ests[k] を refs[k] の推定とみなす（順番の入れ替えはしない）"""
    refs = np.asarray(refs, dtype=float)
    ests = np.asarray(ests, dtype=float)
    out = []
    for k in range(refs.shape[0]):
        t, ei, ea = bss_decomp(refs, ests[k], k, filter_length)
        out.append((_db(t @ t, (ei + ea) @ (ei + ea)), _db(t @ t, ei @ ei), _db((t + ei) @ (t + ei), ea @ ea)))
    return tuple(np.array(v) for v in zip(*out))


def bss_eval(ref, est, filter_length=512, compute_permutation=True):
    """fast_bss_eval.bss_eval_sources の薄い包み。ref・est は (N, n)。
    戻り値 (sdr, sir, sar, perm)：perm[k] は正解 k に対応づけた推定の番号（SIR の和が最大になる順番。
    fast_bss_eval の既定と同じ）。
    compute_permutation=False（順番をそのまま使う）は、fast_bss_eval 0.1.4 と numpy 2 の組み合わせでは
    エラーになる（IV-7 の手順 B）。そこで順番を求めさせ、それが「そのまま」と違うときだけ、自前の
    bss_eval_np で与えられた順番のまま計算し直す（遅いが同じ値）。"""
    import fast_bss_eval
    ref = np.atleast_2d(np.asarray(ref, dtype=float))
    est = np.atleast_2d(np.asarray(est, dtype=float))
    with np.errstate(divide="ignore", invalid="ignore"):
        s, i, a, p = fast_bss_eval.bss_eval_sources(ref, est, filter_length=filter_length,
                                                    compute_permutation=True)
    p = np.asarray(p)
    if not compute_permutation and not np.array_equal(p, np.arange(ref.shape[0])):
        s, i, a = bss_eval_np(ref, est, filter_length)
        p = np.arange(ref.shape[0])
    return np.asarray(s), np.asarray(i), np.asarray(a), p

In [ ]:
%%writefile bss.py
# 第7章の ~/home-lab/ch7/bss.py（AuxIVA・ILRMA の実装）と同じもの
#!/usr/bin/env python3
"""第7章：AuxIVA と ILRMA を NumPy でゼロから実装する（決定系：音源の数 N ＝ マイクの数 M）

記号（論文の周波数・時間の番号 i, j を、この章では f, t と書く。T はフレームの数で、
      論文の基底行列 T_n（ここでは Tb）とは別のもの）
  X : 観測の STFT。形は (F, T, M)。x_ft = [x_ft1, …, x_ftM]ᵀ
  W : 分離行列。形は (F, N, M)。y_ft = W_f x_ft。W_f の n 行目が w_nfᴴ
  Y : 分離信号 (F, T, N)、P = |Y|²
  r : 音源 n の分散（音源モデル）。AuxIVA は r_nt = √(Σ_f |y_nft|²)（周波数をまとめた大きさ）、
      ILRMA は r_nft = (T_n V_n)_ft（音源ごとの NMF）
更新（反復射影法 IP）
  U_nf = (1/T) Σ_t x_ft x_ftᴴ / r_n(ft)     （AuxIVA では φ(r)=1/r を掛ける）
  w_nf ← (W_f U_nf)⁻¹ e_n、 w_nf ← w_nf / √(w_nfᴴ U_nf w_nf)
コスト（負の対数尤度。反復で増えないことを確かめる）
  ILRMA : Σ_n Σ_ft (|y_nft|²/r_nft + log r_nft) − 2T Σ_f log|det W_f|
  AuxIVA: Σ_n Σ_t r_nt − T Σ_f log|det W_f|      （ラプラス分布の音源モデル）
          log|det| の係数が ILRMA の半分なのは、尺度を変えたラプラス分布の負の対数尤度を2で割ったものだから。
          定数倍なので、反復で増えないことの確認には影響しない
"""
import numpy as np

EPS = 1e-10


def demix(W, X):
    """Y = W X（周波数ごと）"""
    return np.einsum("fnm,ftm->ftn", W, X)


def ip_update(W, X, phi, n):
    """反復射影法で W の n 行目を更新する。phi は (F, T) の重み（1/r）"""
    F, T, M = X.shape
    U = np.einsum("ftm,ftk,ft->fmk", X, X.conj(), phi) / T          # (F, M, M)
    WU = W @ U
    e = np.zeros((F, M, 1), dtype=complex)
    e[:, n, 0] = 1.0
    w = np.linalg.solve(WU, e)[:, :, 0]                               # (F, M)
    denom = np.sqrt(np.real(np.einsum("fm,fmk,fk->f", w.conj(), U, w)))
    w = w / (denom[:, None] + EPS)
    W[:, n, :] = w.conj()
    return W


def logdet(W):
    return np.sum(np.log(np.abs(np.linalg.det(W)) + EPS))


def projection_back(Y, W, ref=0):
    """スケールのあいまいさを、基準マイク ref で観測した大きさに合わせて直す"""
    A = np.linalg.inv(W)                                              # (F, M, N) 混合行列の推定
    return Y * A[:, ref, :][:, None, :]


def auxiva(X, n_iter=100, return_cost=False):
    F, T, M = X.shape
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    hist = []
    for _ in range(n_iter):
        for n in range(M):
            Y = demix(W, X)
            r = np.sqrt(np.sum(np.abs(Y[:, :, n]) ** 2, axis=0)) + EPS   # (T,)
            W = ip_update(W, X, np.broadcast_to(1.0 / r, (F, T)), n)
        if return_cost:
            Y = demix(W, X)
            r = np.sqrt(np.sum(np.abs(Y) ** 2, axis=0)) + EPS           # (T, N)
            hist.append(float(np.sum(r) - T * logdet(W)))
    Y = projection_back(demix(W, X), W)
    return (Y, W, np.array(hist)) if return_cost else (Y, W)


def ilrma(X, n_basis=4, n_iter=100, seed=0, return_cost=False):
    F, T, M = X.shape
    rng = np.random.default_rng(seed)
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    Tb = rng.uniform(0.1, 1.0, (M, F, n_basis))                       # 基底 T_n
    Vb = rng.uniform(0.1, 1.0, (M, n_basis, T))                       # アクティベーション V_n
    R = Tb @ Vb                                                       # (N, F, T)
    hist = []
    for _ in range(n_iter):
        P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)               # (N, F, T)
        # 音源モデルの更新（IS ダイバージェンスの NMF。補助関数法の乗法更新）
        Tb *= np.sqrt(((P / R ** 2) @ np.swapaxes(Vb, 1, 2)) / ((1 / R) @ np.swapaxes(Vb, 1, 2) + EPS))
        R = Tb @ Vb + EPS
        Vb *= np.sqrt((np.swapaxes(Tb, 1, 2) @ (P / R ** 2)) / (np.swapaxes(Tb, 1, 2) @ (1 / R) + EPS))
        R = Tb @ Vb + EPS
        # 分離行列の更新（IP）
        for n in range(M):
            W = ip_update(W, X, 1.0 / R[n], n)
        # スケールの正規化（W の各行と、音源モデルの大きさをそろえる。結果は変わらない）
        Y = demix(W, X)
        lam = np.sqrt(np.mean(np.abs(Y) ** 2, axis=(0, 1))) + EPS     # (N,)
        W /= lam[None, :, None]
        R /= lam[:, None, None] ** 2
        Tb /= lam[:, None, None] ** 2
        if return_cost:
            P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
            hist.append(float(np.sum(P / R + np.log(R)) - 2 * T * logdet(W)))
    Y = projection_back(demix(W, X), W)
    return (Y, W, np.array(hist)) if return_cost else (Y, W)

In [ ]:
%%writefile bss6.py
# IV-6 の ~/home-lab/partIV/iv6/bss6.py と同じもの（ファイルの場所の 3 行だけ変えた）
#!/usr/bin/env python3
"""IV-6：第7章の bss.py をそのまま使い、理論の確かめに要る道具を足す（この章の共通部品）

第7章の ~/home-lab/ch7/bss.py を import する（demix・ip_update・logdet・projection_back・auxiva・ilrma）。
配列の形は第7章と同じ：X は (F, T, M)、W は (F, N, M)、y_ft = W_f x_ft、Y は (F, T, N)。

この章で足すもの
  sources(names, fs, sec)          : librosa の例の音源を読み、長さと大きさをそろえる
  simulate(src, fs, rt60, seed, …) : pyroomacoustics の鏡像法で M 本のマイクの観測を作る（rt60=0 は無響）
  fdica(X, n_iter)                 : 周波数ごとに独立な ICA（AuxICA）。パーミュテーションは解かない
  align_power_ratio(Y, W)          : パワー比の相関で順番をそろえる（ブラインド。Sawada らの方法の簡易版）
  align_oracle(Y, ref_img)         : 正解を使って順番をそろえる（上限を見るための oracle）
  swapped_stats(perms, energy_db)  : 順番が多数派と違うビンを、エネルギーのあるビンだけで数える
  ilrma_v(X, K, n_iter, …)         : 第7章の ilrma と同じ計算に、初期値・更新の順番・正規化を選ぶ引数を足したもの
  oracle_w(X, img)                 : 正解の像に最小二乗で合わせた分離行列（周波数ごとの線形分離の上限の目安）
  ip_oracle(X, img, n_iter)        : 音源モデルに正解のパワーを入れた IP（音源モデルが完璧なときの上限）
  cost_ilrma / cost_auxiva         : 負の対数尤度（第7章と同じ式）
  to_time / evaluate               : 逆 STFT と BSS Eval（fast_bss_eval）
"""
import pathlib
import sys

import numpy as np

sys.path.insert(0, ".")   # ノートブック版：common/ と第7章の bss.py は、このノートブックの隣に書き出してある

import bss                                              # noqa: E402  第7章の実装
from bss import demix, ip_update, logdet, projection_back, EPS   # noqa: E402,F401
from common.stft import stft, istft                     # noqa: E402

L_WIN, S_HOP = 2048, 512                                 # 第IV部の分離の標準（16 kHz で 128 ms、第7章と同じ）


# ---------------------------------------------------------------- 音源と部屋
def sources(names=("libri1", "libri3"), fs=16000, sec=8.0, offset=0.0):
    """librosa の例の音源を読み、同じ長さにして標準偏差を 1 にそろえる。(N, n) を返す"""
    import librosa
    s = [librosa.load(librosa.ex(nm), sr=fs, mono=True, offset=offset, duration=sec)[0] for nm in names]
    n = min(len(x) for x in s)
    return np.stack([x[:n] / (np.std(x[:n]) + 1e-12) for x in s]).astype(float)


def simulate(src, fs, rt60, seed, n_mics=2, d=0.04, dist=1.5, min_sep_deg=30):
    """直方体の部屋（幅 5〜8 m・奥行 4〜6 m・高さ 3 m）に、間隔 d [m] の直線アレイと音源を置く。
    音源はアレイの中心から dist [m]、方向は互いに min_sep_deg 度以上離す（第7章の simulate と同じ考え方）。
    戻り値：mix (M, n)、img (N, n)（各音源を 0 番のマイクで録った像＝BSS Eval の正解）"""
    import pyroomacoustics as pra
    rng = np.random.default_rng(seed)
    dim = np.array([rng.uniform(5, 8), rng.uniform(4, 6), 3.0])
    if rt60 > 0:
        e_abs, max_order = pra.inverse_sabine(rt60, dim)
        room = pra.ShoeBox(dim, fs=fs, materials=pra.Material(e_abs), max_order=max_order)
    else:
        room = pra.ShoeBox(dim, fs=fs, max_order=0)        # 反射なし（無響）
    c = np.array([dim[0] / 2 + rng.uniform(-0.5, 0.5), dim[1] / 2 + rng.uniform(-0.5, 0.5), 1.5])
    off = (np.arange(n_mics) - (n_mics - 1) / 2) * d
    room.add_microphone_array(np.stack([c + [o, 0, 0] for o in off], axis=1))
    N = src.shape[0]
    while True:
        ang = np.sort(rng.uniform(0, np.pi, N))
        if N == 1 or np.min(np.diff(ang)) >= np.radians(min_sep_deg):
            break
    for k in range(N):
        room.add_source(c + dist * np.array([np.cos(ang[k]), np.sin(ang[k]), 0.0]), signal=src[k])
    pre = room.simulate(return_premix=True)                # (音源, マイク, 時間)
    n = src.shape[1]
    return pre.sum(axis=0)[:, :n], pre[:, 0, :n]


def bin_hz(mask, fs=16000):
    """True のビンの周波数の範囲を "a〜b Hz" の文字列で返す（表示用）"""
    f = np.flatnonzero(mask) * fs / L_WIN
    return f"{f.min():.0f}〜{f.max():.0f} Hz"


def to_stft(x):
    """(M, n) の時間信号 → (F, T, M) の STFT（common.stft、L=2048・S=512・hann）"""
    return stft(np.asarray(x).T, L_WIN, S_HOP)


def to_time(Y, length):
    """(F, T, N) → (N, length) の時間信号（common.istft。length は必ず元の長さ）"""
    return np.stack([istft(Y[:, :, n], L_WIN, S_HOP, length=length) for n in range(Y.shape[2])])


def evaluate(ref, est):
    """BSS Eval（fast_bss_eval）。順番は正解に合わせて選ぶ。音源ごとの SDR・SIR・SAR を返す"""
    import fast_bss_eval
    with np.errstate(divide="ignore", invalid="ignore"):
        sdr, sir, sar, _ = fast_bss_eval.bss_eval_sources(ref, est, compute_permutation=True)
    return np.asarray(sdr), np.asarray(sir), np.asarray(sar)


# ---------------------------------------------------------------- コスト（第7章と同じ式）
def cost_ilrma(W, X, R):
    """Σ_n Σ_ft (|y|²/r + log r) − 2T Σ_f log|det W_f|。R は (N, F, T)"""
    T = X.shape[1]
    P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
    return float(np.sum(P / R + np.log(R)) - 2 * T * logdet(W))


def cost_auxiva(W, X):
    """第7章の AuxIVA のコスト Σ_n Σ_t r_nt − T Σ_f log|det W_f|（r_nt = √Σ_f |y_nft|²）"""
    T = X.shape[1]
    r = np.sqrt(np.sum(np.abs(demix(W, X)) ** 2, axis=0)) + EPS
    return float(np.sum(r) - T * logdet(W))


# ---------------------------------------------------------------- FDICA とパーミュテーション
def fdica(X, n_iter=100, W0=None, floor=1e-3):
    """周波数ごとに独立な ICA。音源モデルは周波数ごとの複素ラプラス分布 p(y) ∝ exp(−2|y|)、
    つまり AuxIVA の r_nt を r_nft = |y_nft| に変えただけ（周波数どうしをつなぐものが何もない）。
    更新は第7章の ip_update（IP）をそのまま使う。
    |y| が 0 に近いと重み 1/|y| が大きくなりすぎて計算が壊れる（NaN）ので、|y| が floor より小さいところは
    floor に切り上げる。これは「r ≤ floor では 2 乗、それより上では r」のフーバー型の G を使うことにあたり、
    G'(r)/r は増えないので補助関数法が使える。floor は反復の間ずっと同じ定数なので、コスト J も1つに決まり、
    J は反復で増えない。IP の正規化で収束後は平均 |y| ≈ 1 になるので、floor = 1e-3 は「平均の 1000 分の 1」の目安。"""
    F, T, M = X.shape
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1)) if W0 is None else W0.copy()
    for _ in range(n_iter):
        for n in range(M):
            r = np.maximum(np.abs(demix(W, X)[:, :, n]), floor)   # (F, T)
            W = ip_update(W, X, 1.0 / r, n)
    return W


def align_oracle(Y, img_stft):
    """正解の像の STFT img_stft (F, T, N) と、振幅の包絡の相関が最大になる順番を周波数ごとに選ぶ（oracle）"""
    import itertools
    F, T, N = Y.shape
    a = np.abs(Y) - np.abs(Y).mean(axis=1, keepdims=True)
    b = np.abs(img_stft) - np.abs(img_stft).mean(axis=1, keepdims=True)
    a /= np.linalg.norm(a, axis=1, keepdims=True) + 1e-12
    b /= np.linalg.norm(b, axis=1, keepdims=True) + 1e-12
    C = np.einsum("ftn,ftk->fnk", a, b)                    # C[f, 出力 n, 音源 k]
    best = np.zeros((F, N), dtype=int)
    score = np.full(F, -np.inf)
    for p in itertools.permutations(range(N)):             # p[k] = 音源 k に割り当てる出力の番号
        s = sum(C[:, p[k], k] for k in range(N))
        better = s > score
        best[better] = p
        score[better] = s[better]
    return best                                            # Y[:, :, best[f]] が音源の順


def align_power_ratio(Yp, n_rounds=10):
    """ブラインドな順番そろえ：プロジェクションバック後の出力 Yp (F, T, N) のパワー比
    q_nft = |y_nft|² / Σ_m |y_mft|² を「その音源がどれだけ鳴っているか」とみなし、
    全周波数の平均（重心）との相関が最大になる順番を周波数ごとに選ぶ、をくり返す。"""
    import itertools
    F, T, N = Yp.shape
    q = np.abs(Yp) ** 2
    q = q / (q.sum(axis=2, keepdims=True) + 1e-12)
    q = q - q.mean(axis=1, keepdims=True)
    q /= np.linalg.norm(q, axis=1, keepdims=True) + 1e-12
    perms = np.tile(np.arange(N), (F, 1))
    for _ in range(n_rounds):
        cur = np.take_along_axis(q, perms[:, None, :], axis=2)
        cen = cur.mean(axis=0)                             # (T, N) 重心
        C = np.einsum("ftn,tk->fnk", q, cen)
        score = np.full(F, -np.inf)
        new = perms.copy()
        for p in itertools.permutations(range(N)):
            s = sum(C[:, p[k], k] for k in range(N))
            better = s > score
            new[better] = p
            score[better] = s[better]
        if np.array_equal(new, perms):
            break
        perms = new
    return perms


def swapped_stats(perms, energy_db, thr_db=-40.0):
    """2 音源のとき、多いほうの順番と違う周波数を数える。
    正解の像のエネルギーが最大から thr_db より小さいビンは、どちらの順番でも差がほとんどなく
    順番が決まらないので数えない。戻り値：(数えたビンでの割合, 入れ替わったビンの数, エネルギーで重みづけた割合)"""
    p = perms[:, 0]
    use = energy_db > thr_db
    maj = np.bincount(p[use]).argmax()
    sw = (p != maj) & use
    w = 10 ** (energy_db / 10)
    return float(sw.sum() / use.sum()), int(sw.sum()), float(np.sum(w[sw]) / np.sum(w))


def energy_db(img_stft):
    """周波数ごとの正解の像のエネルギー [dB]（最大を 0 dB）"""
    E = np.sum(np.abs(img_stft) ** 2, axis=(1, 2))
    return 10 * np.log10(E / E.max() + 1e-30)


def permute(Y, perms):
    """Y (F, T, N) の出力の順番を周波数ごとに perms (F, N) で並べ替える"""
    return np.take_along_axis(Y, perms[:, None, :], axis=2)


# ---------------------------------------------------------------- ILRMA（第7章と同じ計算＋比べるための引数）
def init_ch7(M, F, T, K, seed):
    """第7章の ilrma と同じ初期値（default_rng(seed) で Tb → Vb の順に一様乱数 0.1〜1.0）"""
    rng = np.random.default_rng(seed)
    return rng.uniform(0.1, 1.0, (M, F, K)), rng.uniform(0.1, 1.0, (M, K, T))


def init_pra(M, F, T, K, seed):
    """pyroomacoustics の ilrma と同じ初期値（np.random.seed(seed) のあと rand で T → V の順。V は転置して返す）"""
    rs = np.random.RandomState(seed)
    Tb = 0.1 + 0.9 * rs.rand(M, F, K)
    Vb = 0.1 + 0.9 * rs.rand(M, T, K)
    return Tb, np.ascontiguousarray(np.swapaxes(Vb, 1, 2))


def ilrma_v(X, n_basis=2, n_iter=100, init=None, seed=0, order="ch7", norm="row", fresh_ir=False,
            return_cost=False, checkpoints=(), on_checkpoint=None):
    """ILRMA。order="ch7" かつ norm="row" かつ init=None なら、第7章の bss.ilrma と同じ計算になる。
      init   : (Tb0, Vb0) を与えるとそれを使う（init_ch7・init_pra で作る）
      order  : "ch7"＝全音源の NMF → 全音源の IP、"pra"＝音源ごとに NMF → IP（pyroomacoustics の順。eps も合わせる）
      fresh_ir : order="pra" のとき、各反復の最初の基底の更新で 1/R を作り直す（True）か、
               pyroomacoustics 0.10.1 と同じく正規化の前の 1/R を使う（False）か
      norm   : "row"＝W の n 行目を割る（正しい）、"col"＝W の n 列目に掛ける（pyroomacoustics 0.10.1 の書き方）、
               None＝正規化しない
      checkpoints と on_checkpoint(i, W) で、途中の反復（i 回目のあと）の W を受け取れる
    戻り値：(Y（プロジェクションバック済み）, W, コストの履歴)"""
    F, T, M = X.shape
    Tb, Vb = init if init is not None else init_ch7(M, F, T, n_basis, seed)
    Tb, Vb = Tb.copy(), Vb.copy()
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    hist = []
    if order == "ch7":
        R = Tb @ Vb
    else:
        eps = 1e-15
        R = np.maximum(Tb @ Vb, eps)
        iR_all = 1 / R                                   # pyroomacoustics は 1/R を別の配列に持つ
        P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
    for it in range(1, n_iter + 1):
        if order == "ch7":
            P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
            Vt = np.swapaxes(Vb, 1, 2)
            Tb *= np.sqrt(((P / R ** 2) @ Vt) / ((1 / R) @ Vt + EPS))
            R = Tb @ Vb + EPS
            Tt = np.swapaxes(Tb, 1, 2)
            Vb *= np.sqrt((Tt @ (P / R ** 2)) / (Tt @ (1 / R) + EPS))
            R = Tb @ Vb + EPS
            for n in range(M):
                W = ip_update(W, X, 1.0 / R[n], n)
        else:                                            # pyroomacoustics の順番と eps の扱い
            for s in range(M):
                iR = 1 / R[s] if fresh_ir else iR_all[s]     # pyroomacoustics は正規化のあとの 1/R を作り直さない
                Tb[s] *= np.sqrt(((P[s] * iR ** 2) @ Vb[s].T) / (iR @ Vb[s].T))
                Tb[Tb < eps] = eps
                R[s] = np.maximum(Tb[s] @ Vb[s], eps)
                iR = iR_all[s] = 1 / R[s]
                Vb[s] *= np.sqrt((Tb[s].T @ (P[s] * iR ** 2)) / (Tb[s].T @ iR))
                Vb[Vb < eps] = eps
                R[s] = np.maximum(Tb[s] @ Vb[s], eps)
                iR_all[s] = 1 / R[s]
                U = np.einsum("ftm,ftk,ft->fmk", X, X.conj(), 1 / R[s]) / T
                e = np.zeros((F, M, 1), dtype=complex)
                e[:, s, 0] = 1.0
                w = np.linalg.solve(W @ U, e)[:, :, 0]
                w = w / np.sqrt(np.real(np.einsum("fm,fmk,fk->f", w.conj(), U, w)))[:, None]
                W[:, s, :] = w.conj()
            P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
        if norm is not None:
            Pn = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
            lam = np.sqrt(np.mean(Pn, axis=(1, 2))) + (EPS if order == "ch7" else 0.0)   # (N,)
            for s in range(M):
                if norm == "row":
                    W[:, s, :] /= lam[s]
                else:                                    # pyroomacoustics 0.10.1：列に掛けている
                    W[:, :, s] /= lam[s]
                R[s] /= lam[s] ** 2
                Tb[s] /= lam[s] ** 2
                if order != "ch7":
                    P[s] /= lam[s] ** 2
        if return_cost:
            hist.append(cost_ilrma(W, X, R))
        if on_checkpoint is not None and it in checkpoints:
            on_checkpoint(it, W.copy())
    Y = projection_back(demix(W, X), W)
    return Y, W, np.array(hist)


# ---------------------------------------------------------------- oracle（上限の目安）
def oracle_w(X, img_stft):
    """W_f = argmin Σ_t ||s_ft − W x_ft||²（s は正解の像）＝ S Xᴴ (X Xᴴ)⁻¹。出力は最初から像の大きさ"""
    Sxx = np.einsum("ftm,ftk->fmk", X, X.conj())
    Ssx = np.einsum("ftn,ftk->fnk", img_stft, X.conj())
    W = np.linalg.solve(np.swapaxes(Sxx, 1, 2), np.swapaxes(Ssx, 1, 2)).swapaxes(1, 2)   # Ssx Sxx⁻¹
    return W


def ip_oracle(X, img_stft, n_iter=20):
    """音源モデル r_nft に正解の像のパワー |s_nft|² を入れて IP だけを回す（音源モデルが完璧な ILRMA）"""
    F, T, M = X.shape
    R = np.moveaxis(np.abs(img_stft) ** 2, 2, 0) + 1e-6 * np.mean(np.abs(img_stft) ** 2)
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    for _ in range(n_iter):
        for n in range(M):
            W = ip_update(W, X, 1.0 / R[n], n)
    return projection_back(demix(W, X), W), W

In [ ]:
%%writefile masks.py
#!/usr/bin/env python3
"""IV-7：時間周波数マスク（正解を使う理想マスク＝oracle）と、その当て方

  ideal_masks(S, X)  → dict（名前 → マスク (N, F, T)）
      S : 正解の像の STFT (N, F, T)（基準マイクで録った各音源。x = Σ_n s_n が成り立つもの）
      X : 混合音の STFT (F, T)
  apply_masks(Mk, X, L, S_hop, length) → (N, length) の時間信号（マスク × 混合音 → 逆 STFT）

マスクの定義（m_nft を混合音 x_ft に掛けて ŝ_nft = m_nft x_ft とする。d_nft = x_ft − s_nft はほかの音源の和）
  IBM  理想バイナリマスク     1[|s| > |d|]                       {0, 1} の中で |s − m x|² を最小にする
  IRM  理想比マスク           (|s|² / Σ_k |s_k|²)^(1/2)          Wang ら（2014）の β = 1/2
  IWM  ウィーナー様マスク     |s|² / Σ_k |s_k|²                  IV-4-3-5 の理想ウィーナーマスク（β = 1）
  IAM  振幅比マスク           min(|s| / |x|, 1)                  位相を考えない。1 で切る
  PSM  位相を考えたマスク     clip(Re(s x*) / |x|², 0, 1)         実数の中で |s − m x|² を最小にする値を [0, 1] に切る
  cRM  複素マスク             s / x（大きさを 1 以下に切る）     切らなければ正解そのもの（上限の意味がない）
注：IRM・IWM の分母 Σ_k |s_k|² が本文の |s|² + |d|² と同じになるのは音源が 2 つのときだけ（d はほかの音源の和なので、
    3 つ以上では |d|² ≠ Σ_{k≠n} |s_k|²）。3 つ以上の Σ_k |s_k|² の形は、IV-4-3-5 の多音源のウィーナーフィルタ λ_n / Σ_k λ_k と同じ。
"""
import pathlib
import sys

import numpy as np

HERE = pathlib.Path(".").resolve()   # ノートブック版：このノートブックのフォルダ（out/ と runs/ もここ）
sys.path.insert(0, ".")   # ノートブック版：common/・bss.py・bss6.py・masks.py・masknet.py はこのノートブックの隣
from common.stft import istft                            # noqa: E402

EPS = 1e-12
MASKS = ("IBM", "IRM", "IWM", "IAM", "PSM", "cRM")


def ideal_masks(S, X, names=MASKS):
    S = np.asarray(S)
    X = np.asarray(X)
    P = np.abs(S) ** 2                                   # (N, F, T)
    Ptot = P.sum(axis=0, keepdims=True) + EPS
    D = X[None] - S                                      # ほかの音源の和
    ax = np.abs(X)[None] + EPS
    out = {}
    for nm in names:
        if nm == "IBM":
            out[nm] = (np.abs(S) > np.abs(D)).astype(float)
        elif nm == "IRM":
            out[nm] = np.sqrt(P / Ptot)
        elif nm == "IWM":
            out[nm] = P / Ptot
        elif nm == "IAM":
            out[nm] = np.minimum(np.abs(S) / ax, 1.0)
        elif nm == "PSM":
            out[nm] = np.clip(np.real(S * np.conj(X)[None]) / ax ** 2, 0.0, 1.0)
        elif nm == "cRM":
            m = S / (X[None] + EPS)
            out[nm] = m / np.maximum(np.abs(m), 1.0)     # 大きさが 1 を超える分だけ縮める（位相はそのまま）
        else:
            raise ValueError(nm)
    return out


def apply_masks(Mk, X, L, S_hop, length):
    """マスク Mk (N, F, T) を混合音 X (F, T) に掛け、逆 STFT して (N, length) を返す（位相は Mk が複素数でなければ混合音のまま）"""
    return np.stack([istft(Mk[n] * X, L, S_hop, length=length) for n in range(Mk.shape[0])])


def stft_error(Mk, S, X):
    """STFT の上での相対誤差 Σ|s − m x|² / Σ|s|²（dB）。マスクの定義の「最小にしたもの」を数値で比べる"""
    e = np.sum(np.abs(S - Mk * X[None]) ** 2)
    return 10 * np.log10(e / np.sum(np.abs(S) ** 2))

In [ ]:
%%writefile masknet.py
#!/usr/bin/env python3
"""IV-7：小さなマスク推定ネットワーク（BLSTM）、PIT の損失、学習用の混合音の作り方

  MaskNet(n_freq=257, hidden=128, layers=2, n_src=2)   入力：混合音の対数パワー (B, T, F) → マスク (B, N, F, T)
  torch_stft(x, L, S)                                 common.stft と同じ決め方の STFT（torch.stft、中心合わせ・0 詰め）
  pit_loss(est, ref, pair_loss)                       発話単位の PIT（uPIT）：N! 通りの対応のうち、損失が最小のもの
  pit_loss_bruteforce(est, ref, pair_loss)            同じものを素朴に書いたもの（照合用）
  mse_pair(m, t) / psa_pair(...)                      マスクの 2 乗誤差（IRM を目標）／位相を考えた信号の近似
  Mixer(speech, rooms, seg, rng)                      学習・検証の混合音を作る（numpy）。make(B) → (mix (B, n), img (B, 2, n))
  features(X) / irm_target(Simg) / apply_mask(...)    特徴量と目標
使い方（確かめ）: uv run python masknet.py   … torch.stft と common.stft の一致、PIT の照合、形の確認
"""
import itertools
import math
import pathlib
import sys

import numpy as np
import torch
import torch.nn as nn

HERE = pathlib.Path(".").resolve()   # ノートブック版：このノートブックのフォルダ（out/ と runs/ もここ）
sys.path.insert(0, ".")   # ノートブック版：common/・bss.py・bss6.py・masks.py・masknet.py はこのノートブックの隣

L_WIN, S_HOP = 512, 128                                  # 16 kHz で 32 ms・8 ms（マスク推定でよく使う大きさ）
FS = 16000


# ---------------------------------------------------------------- STFT（common.stft と同じ決め方）
def torch_stft(x, L=L_WIN, S=S_HOP):
    """x (B, n) → (B, F, T) の複素数。窓の中心が n = tS、前後に L/2 の 0 を足す（pad_mode="constant"）。
    torch.stft の既定 pad_mode は "reflect" なので、必ず "constant" を指定する（common.stft・第7章と合わせる）"""
    w = torch.hann_window(L, periodic=True, dtype=x.dtype, device=x.device)
    return torch.stft(x, n_fft=L, hop_length=S, win_length=L, window=w, center=True,
                      pad_mode="constant", return_complex=True)


def features(X, eps=1e-8):
    """混合音の STFT (B, F, T) → 対数パワー (B, T, F)。大きさの違いは Mixer で混合音の標準偏差を 1 にしてそろえる"""
    return torch.log10(X.abs() ** 2 + eps).transpose(1, 2)


def irm_target(Simg, eps=1e-8):
    """正解の像の STFT (B, N, F, T) → IRM (B, N, F, T) = (|s_n|² / Σ_k |s_k|²)^(1/2)"""
    P = Simg.abs() ** 2
    return torch.sqrt(P / (P.sum(dim=1, keepdim=True) + eps))


# ---------------------------------------------------------------- ネットワーク
class MaskNet(nn.Module):
    """2 層の双方向 LSTM（BLSTM）＋全結合＋シグモイド。音源ごとに 0〜1 のマスクを出す"""

    def __init__(self, n_freq=L_WIN // 2 + 1, hidden=128, layers=2, n_src=2, dropout=0.0):
        super().__init__()
        self.n_freq, self.n_src = n_freq, n_src
        self.norm = nn.LayerNorm(n_freq)                 # 周波数方向の大きさをそろえる（学習を安定させる）
        self.lstm = nn.LSTM(n_freq, hidden, layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.fc = nn.Linear(2 * hidden, n_src * n_freq)

    def forward(self, feat):                             # feat (B, T, F)
        h, _ = self.lstm(self.norm(feat))
        m = torch.sigmoid(self.fc(h))                    # (B, T, N·F)
        B, T, _ = m.shape
        return m.view(B, T, self.n_src, self.n_freq).permute(0, 2, 3, 1)   # (B, N, F, T)


# ---------------------------------------------------------------- PIT
def mse_pair(m_est, m_ref):
    """1 組（推定 1 つと正解 1 つ）のマスクの 2 乗誤差。(B, F, T) → (B,)"""
    return ((m_est - m_ref) ** 2).mean(dim=(-2, -1))


def pit_loss(est, ref, pair_loss=mse_pair):
    """発話単位の PIT。est・ref は (B, N, …)。
    C[b, i, j] = pair_loss(est_i, ref_j) を N² 回だけ計算し、N! 通りの対応 π の Σ_j C[b, π(j), j] / N の最小を選ぶ。
    戻り値：(バッチの平均の損失, 選んだ対応 (B, N)：perm[b, j] は正解 j に当てた推定の番号)"""
    B, N = est.shape[:2]
    C = torch.stack([torch.stack([pair_loss(est[:, i], ref[:, j]) for j in range(N)], dim=1) for i in range(N)], dim=1)
    perms = torch.tensor(list(itertools.permutations(range(N))), device=est.device)   # (P, N)
    cols = torch.arange(N, device=est.device)
    tot = torch.stack([C[:, p, cols].mean(dim=1) for p in perms], dim=1)              # (B, P)
    best, idx = tot.min(dim=1)
    return best.mean(), perms[idx]


def pit_loss_bruteforce(est, ref, pair_loss=mse_pair):
    """照合用：発話ごと・対応ごとに素朴に計算する（遅い）"""
    B, N = est.shape[:2]
    out = []
    for b in range(B):
        vals = [sum(float(pair_loss(est[b:b + 1, p[j]], ref[b:b + 1, j])) for j in range(N)) / N
                for p in itertools.permutations(range(N))]
        out.append(min(vals))
    return float(np.mean(out))


def psa_pair_factory(Xmag, Xph):
    """位相を考えた信号の近似（PSA、Erdogan ら 2015）の 1 組の損失を作る。
    目標 t = clip(|s| cos(∠s − ∠x), 0, |x|) と m|x| の 2 乗誤差（混合音のパワーで割って大きさをそろえる）"""
    def pair(m_est, S_ref):
        t = torch.clamp(S_ref.abs() * torch.cos(torch.angle(S_ref) - Xph), min=0.0)
        t = torch.minimum(t, Xmag)
        return ((m_est * Xmag - t) ** 2).mean(dim=(-2, -1)) / (Xmag ** 2).mean(dim=(-2, -1))
    return pair


# ---------------------------------------------------------------- 学習用の混合音（numpy）
SPEEDS = ((10, 9), (20, 19), (1, 1), (20, 21), (10, 11))   # 長さが 10/9 倍〜10/11 倍＝ゆっくり〜速く（高さも変わる）
class Mixer:
    """speech.npz と rooms_*.npz から、2 人の声の混合音を作る。
    1 つの例：部屋を 1 つ、話者を 2 人（違う人）選び、それぞれの区間から seg 秒を切り出し、
    相対レベルを ±5 dB の一様乱数で変え、部屋のインパルス応答（マイク 0 番）をたたみ込んで像を作り、足して混合音にする。
    最後に、混合音の標準偏差が 1 になるように、混合音と像を同じ数で割る。"""

    def __init__(self, speech_npz, rooms_npz, split, seg=2.0, rng=None, speed=False):
        self.speed = speed                                # True：速さを 0.9〜1.1 倍に変える（データ拡張）
        sp = np.load(speech_npz)
        self.sp = {k.split("_", 1)[1]: sp[k] for k in sp.files if k.startswith(split + "_")}
        self.names = sorted(self.sp)
        r = np.load(rooms_npz)
        self.rir = r["rir"]
        self.n = int(seg * FS)
        self.rng = rng if rng is not None else np.random.default_rng(0)

    def crop(self, x):
        up, down = (1, 1)
        if self.speed:                                    # 速さと高さを少し変える（resample_poly で up/down 倍の長さに）
            up, down = SPEEDS[self.rng.integers(len(SPEEDS))]
        m = int(math.ceil(self.n * down / up))            # 変換の後で n 標本になるだけ切り出す
        if len(x) <= m:
            seg = np.pad(x, (0, m - len(x)))
        else:
            st = self.rng.integers(0, len(x) - m + 1)
            seg = x[st:st + m]
        if (up, down) != (1, 1):
            from scipy.signal import resample_poly
            seg = resample_poly(seg, up, down)
        return seg[:self.n]

    def make(self, B):
        from scipy.signal import fftconvolve
        mix = np.zeros((B, self.n), dtype=np.float32)
        img = np.zeros((B, 2, self.n), dtype=np.float32)
        for b in range(B):
            a, c = self.rng.choice(len(self.names), 2, replace=False)
            room = self.rng.integers(len(self.rir))
            g = 10 ** (self.rng.uniform(-5, 5) / 20)
            for k, (nm, gain) in enumerate(((self.names[a], 1.0), (self.names[c], g))):
                img[b, k] = fftconvolve(gain * self.crop(self.sp[nm]), self.rir[room, k])[:self.n]
            mix[b] = img[b].sum(axis=0)
            sc = mix[b].std() + 1e-8
            mix[b] /= sc
            img[b] /= sc
        return mix, img


def pick_device(name="auto"):
    """cuda → mps → cpu の順に探す（III-4 の repro.py と同じ考え方）"""
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def self_check():
    from common.stft import stft
    rng = np.random.default_rng(0)
    x = rng.standard_normal((2, 16000))
    Xt = torch_stft(torch.from_numpy(x)).numpy()
    Xn = np.stack([stft(x[b], L_WIN, S_HOP) for b in range(2)])
    print(f"[1] torch.stft と common.stft：形 {Xt.shape} と {Xn.shape}、差の最大 {np.max(np.abs(Xt - Xn)):.1e}（float64）")
    Xr = torch.stft(torch.from_numpy(x), n_fft=L_WIN, hop_length=S_HOP, window=torch.hann_window(L_WIN, dtype=torch.float64),
                    center=True, return_complex=True).numpy()
    print(f"    pad_mode を既定（reflect）のままにすると、最初のフレームの差 {np.max(np.abs(Xr[:, :, 0] - Xn[:, :, 0])):.2f}、"
          f"中ほどのフレームの差 {np.max(np.abs(Xr[:, :, 50] - Xn[:, :, 50])):.1e}")
    torch.manual_seed(0)
    est = torch.rand(4, 3, 5, 7)
    ref = torch.rand(4, 3, 5, 7)
    l1, perm = pit_loss(est, ref)
    l2 = pit_loss_bruteforce(est, ref)
    print(f"[2] PIT（N = 3、6 通り）：行列で {float(l1):.10f}、素朴に {l2:.10f}、差 {abs(float(l1) - l2):.1e}")
    l3, _ = pit_loss(est, ref[:, [2, 0, 1]])
    print(f"    正解の順番を入れ替えても同じ：{float(l3):.10f}。固定した順番の損失 {float(mse_pair(est, ref).mean()):.6f} 以下")
    try:
        import fast_bss_eval
        if fast_bss_eval.has_torch:
            s = torch.randn(3, 2, 4000, dtype=torch.float64)
            e = s[:, [1, 0]] + 0.3 * torch.randn(3, 2, 4000, dtype=torch.float64)

            def neg_si(a, b):
                al = (a * b).sum(-1) / (b * b).sum(-1)
                t = al[..., None] * b
                return -10 * torch.log10((t ** 2).sum(-1) / ((t - a) ** 2).sum(-1))
            mine, p = pit_loss(e, s, neg_si)
            ref_loss = fast_bss_eval.si_sdr_pit_loss(s, e).mean()
            print(f"[3] PIT＋負の SI-SDR：自前 {float(mine):.10f}、fast_bss_eval.si_sdr_pit_loss {float(ref_loss):.10f}、"
                  f"選んだ対応 {p[0].tolist()}（入れ替えた正解を見つけた）")
    except ImportError:
        print("[3] fast_bss_eval が無いので省略")
    m = MaskNet()
    f = features(torch_stft(torch.randn(2, 32000)))
    print(f"[4] MaskNet：パラメータ {n_params(m):,} 個、入力 {tuple(f.shape)} → 出力 {tuple(m(f).shape)}、"
          f"使える device：{pick_device()}")


if __name__ == "__main__":
    self_check()

In [ ]:
import sys, re, warnings, logging
sys.path.insert(0, ".")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告を出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
warnings.filterwarnings("ignore", message="Glyph")
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from common import plotting                      # import すると Hiragino Sans が設定される
from common import metrics
import bss6, bss, masks, masknet                 # bss6 は第7章の bss.py を読み込む
matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # 本文の候補（0-1 のセルで用意したフォントが使われる）
matplotlib.rcParams["axes.unicode_minus"] = False
print("common.metrics・bss6・masks・masknet を読み込めました。計算の場所（device）：", masknet.pick_device())


def listen_file(path):
    """保存した wav をそのままの大きさで再生ボタンにする。normalize=False を消さない"""
    import numpy as np, soundfile   # 各節のプログラムが np などの名前を使うので、ここで読み直す
    x, fs = soundfile.read(path)
    assert np.max(np.abs(x)) <= 0.32, "大きすぎる音は鳴らさない"
    display(Audio(x.T if x.ndim > 1 else x, rate=fs, normalize=False))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, i=1, text=None):
    """直前のセルの出力（cap.stdout）から、pattern の (…) の数を取り出す"""
    import re
    m = re.search(pattern, cap.stdout if text is None else text)
    assert m, "出力に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))   # 出力の「−」（全角のマイナス）も数にする


_ok, _num = ok, num   # 各節のプログラムが ok などの名前を使うことがあるので、確かめのセルで戻す

## 1. 評価指標を自分で実装して照合し、落とし穴を確かめる（IV-7-3-2〜3-4、手順 B・C、`metrics_check.py`）

1〜2 分かかります。[1]・[2] が手順 B（自前の SDR・SI-SDR と fast_bss_eval・mir_eval の照合）、[3]〜[8] が手順 C（尺度・フィルタ長・参照信号・短い区間・平均のとり方・改善量の基準）です。3-4 の表と見比べてください。

[2] の最初の行は、ライブラリの挙動の確かめです。fast_bss_eval 0.1.4 と numpy 2 では `compute_permutation=False` がエラーになります（版が違えば動くかもしれません。動いた／動かなかったと版をノートに書いてください）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-7：評価指標を自分で実装して照合し、落とし穴を数値で確かめる

使い方: cd ~/home-lab/partIV/iv7 && uv run python metrics_check.py
  [1] 小さな例の手計算（SI-SDR・SDR）と、行列で直接解いた射影（テプリッツの式の確認）
  [2] 自前（common/metrics.py）と fast_bss_eval・mir_eval の照合（声の部屋のシミュレーション）
  [3] 尺度：推定を 0.5 倍・2 倍・−1 倍にしたときの SNR・SI-SDR・SDR
  [4] 許す歪みのフィルタ長：遅れ・反射を足した推定と、フィルタ長 1〜2048 での SDR
  [5] 参照信号の選び方：同じ推定を「像（残響あり）」と「乾いた音源」で評価する
  [6] 短い信号での不安定さ：真の SNR 0 dB の推定の SDR が、長さでどれだけ上ぶれするか（理論との比較）
  [7] 平均の dB と dB の平均、無音の正解
  [8] 改善量（SDR improvement）の基準
計算時間の目安：1〜2 分（Apple シリコン の MacBook Air、執筆時）
"""
import pathlib
import sys
import time
import warnings

import numpy as np
from scipy.linalg import toeplitz
from scipy.signal import fftconvolve

HERE = pathlib.Path(".").resolve()   # ノートブック版：このノートブックのフォルダ（out/ と runs/ もここ）
sys.path.insert(0, ".")   # ノートブック版：common/・bss.py・bss6.py・masks.py・masknet.py はこのノートブックの隣
import bss6                                              # noqa: E402（中で第7章の bss.py も読む）
from bss import auxiva, demix, projection_back           # noqa: E402
from common import metrics as M                          # noqa: E402

FS = 16000


def direct_sdr(s, e, L):
    """テプリッツの式を使わずに、遅らせた s を列に並べた行列 A で最小二乗を解く（小さな例の確認用）"""
    n = len(s)
    A = np.zeros((n + L - 1, L))
    for k in range(L):
        A[k:k + n, k] = s
    ep = np.concatenate([e, np.zeros(L - 1)])
    h, *_ = np.linalg.lstsq(A, ep, rcond=None)
    t = A @ h
    return 10 * np.log10(t @ t / ((ep - t) @ (ep - t))), h


def part1():
    print("[1] 小さな例")
    s = np.array([1.0, 2.0, -1.0, 0.0, 1.0])
    e = np.array([1.0, 0.0, 0.0, 1.0, 0.0])                # このままでは s との内積が 1（直交していない）
    e = e - (e @ s) / (s @ s) * s                          # s の方向の成分を引いて、s と直交にする
    est = 0.5 * s + e
    print(f"  ‖s‖² = {s @ s:.4f}、e を s と直交にすると ‖e‖² = {e @ e:.4f}")
    print(f"  est = 0.5 s + e：SI-SDR（自前）= {M.si_sdr(s, est):.4f} dB、"
          f"式 10log10(0.25·‖s‖²/‖e‖²) = {10 * np.log10(0.25 * (s @ s) / (e @ e)):.4f} dB、"
          f"SNR = {M.snr(s, est):.4f} dB")
    for L in (1, 2, 3):
        d, h = direct_sdr(s, est, L)
        print(f"  L = {L}：SDR（テプリッツ）= {M.sdr(s, est, L):.6f} dB、行列で直接 = {d:.6f} dB、h = {np.round(h, 4)}")
    rng = np.random.default_rng(0)
    s = rng.standard_normal(400)
    est = fftconvolve(s, [0.0, 0.8, 0.0, -0.3])[:400] + 0.3 * rng.standard_normal(400)
    d, _ = direct_sdr(s, est, 16)
    print(f"  乱数 400 標本、L = 16：テプリッツ {M.sdr(s, est, 16):.10f}、直接 {d:.10f}（差 {abs(M.sdr(s, est, 16) - d):.1e}）")
    print(f"  SI-SDR と L = 1 の SDR：{M.si_sdr(s, est):.10f} と {M.sdr(s, est, 1):.10f}（同じもの）")


def part2(img, est):
    import fast_bss_eval
    print("\n[2] 自前と fast_bss_eval・mir_eval の照合（img・est は (2, n)、8 秒、16 kHz）")
    t0 = time.time()
    own = M.bss_eval_np(img, est)
    t_own = time.time() - t0
    t0 = time.time()
    fb = M.bss_eval(img, est)                                # est は正解の順に並べてあるので perm は [0 1]
    t_fb = time.time() - t0
    try:
        fast_bss_eval.bss_eval_sources(img, est, compute_permutation=False)
        print("  fast_bss_eval の compute_permutation=False：動いた")
    except ValueError as err:
        print(f"  fast_bss_eval の compute_permutation=False はエラー：{str(err)[:60]}…（numpy {np.__version__}）")
    print(f"  自前 SDR {np.round(own[0], 4)}  SIR {np.round(own[1], 4)}  SAR {np.round(own[2], 4)}（{t_own:.1f} 秒）")
    print(f"  fast  SDR {np.round(fb[0], 4)}  SIR {np.round(fb[1], 4)}  SAR {np.round(fb[2], 4)}（{t_fb:.2f} 秒）")
    for nm, a, b in zip(("SDR", "SIR", "SAR"), own, fb[:3]):
        print(f"  {nm} の差の最大 {np.max(np.abs(a - b)):.1e} dB")
    print(f"  sdr()（自前、SDR だけ）{np.round(M.sdr(img, est), 4)}、fast_bss_eval.sdr {np.round(fast_bss_eval.sdr(img, est), 4)}")
    try:
        fsi = fast_bss_eval.si_sdr(img, est)
        how = "fast_bss_eval.si_sdr"
    except AttributeError:                                   # 0.1.4 は torch が無い環境で si_sdr が動かない
        fsi = fast_bss_eval.sdr(img, est, filter_length=1)
        how = "fast_bss_eval.sdr(filter_length=1)（torch が無いので代わりに）"
    print(f"  si_sdr()（自前）{np.round(M.si_sdr(img, est), 6)}、{how} {np.round(fsi, 6)}、"
          f"差の最大 {np.max(np.abs(M.si_sdr(img, est) - fsi)):.1e} dB")
    try:
        import mir_eval
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            t0 = time.time()
            me = mir_eval.separation.bss_eval_sources(img, est, compute_permutation=False)
        print(f"  mir_eval {mir_eval.__version__}：SDR {np.round(me[0], 4)}  SIR {np.round(me[1], 4)}  SAR {np.round(me[2], 4)}"
              f"（{time.time() - t0:.1f} 秒）、自前との差の最大 {np.max(np.abs(np.array(me[:3]) - np.array(own))):.1e} dB")
    except ImportError:
        print("  mir_eval は入っていないので省略（uv add mir_eval で入る。0.8 で bss_eval は非推奨）")
    sw = M.bss_eval(img, est[::-1])
    print(f"  推定の順番を逆にして渡す → fast_bss_eval の perm = {sw[3]}、SDR {np.round(sw[0], 4)}（並べ直してくれる）")


def part3(s, e):
    print("\n[3] 尺度（推定 est を c 倍する）")
    print("     c     SNR    SI-SDR   SDR(512)")
    for c in (1.0, 0.5, 2.0, -1.0):
        print(f"  {c:5.1f}  {M.snr(s, c * e):7.2f}  {M.si_sdr(s, c * e):7.2f}  {M.sdr(s, c * e):7.2f}")


def part4(s):
    print("\n[4] 許す歪みのフィルタ長（正解 s を少し変えただけの推定）")
    n = len(s)
    rng = np.random.default_rng(1)
    noise = 0.01 * np.std(s) * rng.standard_normal(n)       # 40 dB 下の雑音（∞ dB を避ける）
    cases = {
        "そのまま": s,
        "100 標本遅れ（6.25 ms）": np.concatenate([np.zeros(100), s[:-100]]),
        "600 標本遅れ（37.5 ms）": np.concatenate([np.zeros(600), s[:-600]]),
        "反射を1つ足す（3 ms 後に 0.5 倍）": s + 0.5 * np.concatenate([np.zeros(48), s[:-48]]),
        "高い音を弱める（3 タップの平均）": fftconvolve(s, np.ones(3) / 3)[:n],
    }
    Ls = (1, 16, 128, 512, 2048)
    print("  推定 \\ L" + "".join(f"{L:>9d}" for L in Ls) + "   SI-SDR")
    for nm, x in cases.items():
        x = x + noise
        print(f"  {nm:22s}" + "".join(f"{M.sdr(s, x, L):9.2f}" for L in Ls) + f"  {M.si_sdr(s, x):7.2f}")


def part5(src, rts=(0.15, 0.3)):
    import pyroomacoustics as pra
    print("\n[5] 参照信号の選び方（推定 = 音源 0 の像そのもの。完全に分離できたが残響は残っている）")
    for rt in rts:
        _, img = bss6.simulate(src, FS, rt, 0)
        est = img[0] + 1e-3 * np.std(img[0]) * np.random.default_rng(0).standard_normal(img.shape[1])
        dry = src[0]
        print(f"  RT60 {rt:.2f} s：像で評価 SDR {M.sdr(img[0], est):6.2f} dB ／ 乾いた音源で評価 "
              f"SDR(512) {M.sdr(dry, est):6.2f} dB、SDR(2048) {M.sdr(dry, est, 2048):6.2f} dB、SI-SDR {M.si_sdr(dry, est):6.2f} dB")
    print(f"  （512 標本は {512 / FS * 1000:.0f} ms。残響はそれより長く続くので、乾いた音源を正解にすると残響が「ゆがみ」に数えられる）")


def part6(s_long, reps=40):
    print("\n[6] 短い信号での上ぶれ（推定 = 正解 + 白色雑音、真の SNR（‖s‖²/‖e‖²）は 0 dB）")
    rng = np.random.default_rng(2)
    L = 512
    print("    長さ n           理論（白色の正解）  正解が白色雑音      正解が声          SI-SDR（声）")
    for n in (1024, 1600, 4000, 16000, 64000):
        v, w, z = [], [], []
        for _ in range(reps):
            st = rng.integers(0, len(s_long) - n)
            for s, out in ((s_long[st:st + n], v), (rng.standard_normal(n), w)):
                e = rng.standard_normal(n)
                e *= np.linalg.norm(s) / np.linalg.norm(e)
                out.append(M.sdr(s, s + e, L))
                if out is v:
                    z.append(M.si_sdr(s, s + e))
        q = L / n * (1 - (L - 1) / (2 * n))                  # 雑音のうち「許す歪み」の空間に入る割合
        th = 10 * np.log10((1 + q) / (1 - q))
        print(f"  {n:6d}（{n / FS:5.2f} s）   {th:6.2f} dB        {np.mean(w):5.2f} ± {np.std(w, ddof=1):4.2f} dB"
              f"   {np.mean(v):5.2f} ± {np.std(v, ddof=1):4.2f} dB   {np.mean(z):6.2f} dB")


def part7():
    print("\n[7] 平均の dB と dB の平均")
    v = np.array([20.0, 5.0, -10.0])
    print(f"  3 つの試行の SDR {v} dB：dB の平均 {v.mean():.2f} dB、比（パワー）を平均してから dB "
          f"{10 * np.log10(np.mean(10 ** (v / 10))):.2f} dB、中央値 {np.median(v):.2f} dB")
    s = np.zeros(1600)
    e = 1e-3 * np.random.default_rng(3).standard_normal(1600)
    with np.errstate(all="ignore"):
        print(f"  正解が無音（全部 0）のとき：SI-SDR = {M.si_sdr(s, e)}、SNR = {M.snr(s, e):.1f} dB（平均に入れると平均が壊れる）")


def make_estimate(src):
    """RT60 0.3 s・種 0 の部屋で AuxIVA（第7章）を動かし、正解の順番に並べた推定を返す"""
    mix, img = bss6.simulate(src, FS, 0.3, 0)
    n = mix.shape[1]
    y = bss6.to_time(auxiva(bss6.to_stft(mix), 100)[0], n)
    out = M.bss_eval(img, y)
    return mix, img, y[out[3]], out


def part8(mix, img, est, out):
    print("\n[8] 改善量の基準（RT60 0.3 s、種 0、AuxIVA）")
    sdr_in0 = M.sdr(img, np.stack([mix[0], mix[0]]))
    sdr_in1 = M.sdr(img, np.stack([mix[1], mix[1]]))
    print(f"  基準マイク 0 の混合音の SDR（入力 SDR）：{np.round(sdr_in0, 2)} dB、マイク 1 の混合音：{np.round(sdr_in1, 2)} dB")
    print(f"  分離後の SDR {np.round(out[0], 2)} dB → 改善量（マイク 0 基準）{np.round(out[0] - sdr_in0, 2)} dB")
    print(f"  SI-SDR：入力 {np.round(M.si_sdr(img, np.stack([mix[0], mix[0]])), 2)}、出力 {np.round(M.si_sdr(img, est), 2)} dB")


def main():
    src = bss6.sources()                                     # libri1・libri3、8 秒、標準偏差 1
    part1()
    mix, img, est, out = make_estimate(src)                  # [2]〜[4]・[8] で使う推定（AuxIVA の出力）
    part2(img, est)
    part3(img[0], est[0])
    part4(img[0])
    part5(src)
    part6(src[0])
    part7()
    part8(mix, img, est, out)


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
sd = [num(rf"L = {L}：SDR（テプリッツ）= ([-\d.]+) dB") for L in (1, 2, 3)]
ok(all(abs(a - b) < 1e-3 for a, b in zip(sd, (-0.258, 0.430, 4.772))), f"[1] L = 1・2・3 の SDR が −0.258・0.430・4.772 dB と増える {sd}")
ok(num(r"L = 16：テプリッツ [-\d.]+、直接 [-\d.]+（差 ([\d.e+-]+)）") < 1e-12, "[1] テプリッツの式と行列で直接解いた値の差が 1e-15 程度")
a, b = num(r"SI-SDR と L = 1 の SDR：([-\d.]+) と"), num(r"SI-SDR と L = 1 の SDR：[-\d.]+ と ([-\d.]+)")
ok(abs(a - b) < 1e-8, f"[1] SI-SDR と L = 1 の SDR が同じ値（{a}）")
d = {k: num(rf"{k} の差の最大 ([\d.e+-]+) dB") for k in ("SDR", "SIR", "SAR")}
ok(d["SDR"] < 1e-10 and d["SIR"] < 1e-6 and d["SAR"] < 1e-6, f"[2] 自前と fast_bss_eval の差が SDR で 1e-12 程度、SIR・SAR で 1e-7 程度（許容 1e-6 dB）{d}")
ok(num(r"si_sdr\(\)（自前）.*?差の最大 ([\d.e+-]+) dB") < 1e-10, "[2] SI-SDR も自前とライブラリが一致")
if "mir_eval は入っていない" not in cap.stdout:
    ok(num(r"mir_eval .*?自前との差の最大 ([\d.e+-]+) dB") < 1e-6, "[2] mir_eval との差も 1e-7 程度（許容 1e-6 dB）")
s1, s2 = num(r"自前 SDR \[([-\d.]+)"), num(r"自前 SDR \[[-\d.]+\s+([-\d.]+)\]")
ok(abs(s1 - 3.91) < 0.05 and abs(s2 - 5.58) < 0.05, f"[2] SDR が 3.91・5.58 dB 前後（{s1}・{s2}）")
ok("compute_permutation=False：動いた" in cap.stdout or "compute_permutation=False はエラー" in cap.stdout,
   "[2] compute_permutation=False が動いたか動かなかったかが表示された（版とともにノートに書く）")
ok("perm = [1 0]" in cap.stdout, "[2] 推定を逆の順に渡すと perm = [1 0] で並べ直される")
r3 = [tuple(map(float, m)) for m in re.findall(r"^\s+(-?[\d.]+)\s+(-?[\d.]+)\s+(-?[\d.]+)\s+(-?[\d.]+)$", cap.stdout.split("[3]")[1].split("[4]")[0], re.M)]
ok(len(r3) == 4 and all(abs(r[2] - 3.12) < 0.02 and abs(r[3] - 3.91) < 0.02 for r in r3) and abs(r3[0][1] - 4.71) < 0.05 and abs(r3[-1][1] + 5.27) < 0.05,
   f"[3] c を変えても SI-SDR（3.12）と SDR（3.91）は同じで、SNR だけが 4.71 → −5.27 dB と変わる {r3}")
row = [float(v) for v in re.search(r"100 標本遅れ（6\.25 ms）\s+(.*)$", cap.stdout, re.M).group(1).split()]
ok(abs(row[-1] + 19.7) < 0.1 and abs(row[3] - 39.95) < 0.1, f"[4] 100 標本の遅れで SI-SDR −19.7 dB、SDR（L = 512）39.95 dB（{row[-1]}・{row[3]}）")
dry = num(r"RT60 0\.30 s：像で評価 SDR\s+[\d.]+ dB ／ 乾いた音源で評価 SDR\(512\)\s+([-\d.]+) dB")
ok(8 < dry < 10, f"[5] 乾いた音源を正解にすると、RT60 0.3 s で SDR 9 dB 前後（{dry}）")
up = {int(m.group(1)): float(m.group(2)) for m in re.finditer(r"^\s+(\d+)（\s*[\d.]+ s）\s+[\d.]+ dB\s+[-\d.]+ ± [\d.]+ dB\s+([-\d.]+) ±", cap.stdout, re.M)}
ok(1.5 < up[1600] < 2.6 and abs(up[64000]) < 0.2, f"[6] n = 1600 で 2 dB 前後の上ぶれ（{up[1600]}）、n = 64000 でほぼ 0（{up[64000]}）")
ok(num(r"dB の平均 ([\d.]+) dB") == 5.0 and num(r"比（パワー）を平均してから dB ([\d.]+) dB") == 15.37, "[7] 5.00 と 15.37 dB")
imp = re.search(r"改善量（マイク 0 基準）\[([-\d.]+)\s+([-\d.]+)\]", cap.stdout).groups()
ok(abs(float(imp[0]) - 4.55) < 0.05 and abs(float(imp[1]) - 4.66) < 0.05, f"[8] 改善量 4.55・4.66 dB（{imp}）")

**手順 C の 2（自分で予想してから確かめる）**：$L = 2048$ に変えると、$n = 16000$ の上ぶれはいくつになるかを 3-4 の式で計算してから、上のセルの `part6` の `L = 512` を `2048` に書きかえて実行してください（式では $q = 0.128\times(1 - 2047/32000) = 0.1198$、$10\log_{10}(1.1198/0.8802) = 1.05$ dB）。書きかえたら、元に戻してから次へ進んでください。

## 2. 理想マスクの上限を測り、聞く（IV-7-3-6・3-7、手順 D、`oracle_masks.py`）

まず `--seeds 1`（2 試行）で動くことを確かめます（1 分ほど）。結果は `out/oracle_try.csv` に書き、本番とは分けます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['oracle_masks.py', '--seeds', '1', '--out', 'out/oracle_try.csv']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-7：理想マスク（oracle）の上限を、音源分離の論文の作法で ILRMA と並べる

使い方: cd ~/home-lab/partIV/iv7 && uv run python oracle_masks.py [--rt60 0.15 0.3] [--seeds 10]
  部屋・音源・種は IV-6 の eval_iv6.py と同じ（bss6.simulate、libri1・libri3、8 秒、16 kHz）。
  理想マスクは基準マイク（0 番）の混合音だけに掛ける（1 チャネル）。窓は窓長 512（32 ms、シフト 128）と
  窓長 2048（128 ms、シフト 512）の 2 通り。ILRMA（第7章と同じ計算、K = 2、100 回）と oracleW は 2 チャネル。
  [0] STFT の上でのマスクの誤差（定義のとおりの順になるか）
  [1] 種 × 残響の条件ごとの SDR 改善量・SIR・SAR・SI-SDR 改善量（平均 ± 標準偏差。既定は種 10 通り × 2 条件）
出力：--out の CSV（既定 out/oracle_iv7.csv。1 行 ＝ 条件 × 種 × 方法、2 音源の平均）、同じ名前の .png（棒グラフ）、
      同じフォルダの <CSV の名前>_listen_*.wav（RT60 0.3 s・種 0。最大 −10 dBFS。既定では out/oracle_iv7_listen_*.wav）
計算時間の目安：20 試行で 10〜15 分（Apple シリコン の MacBook Air、執筆時）
"""
import argparse
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する

import masks
HERE = pathlib.Path(".").resolve()   # ノートブック版：このノートブックのフォルダ（out/ と runs/ もここ）
sys.path.insert(0, ".")   # ノートブック版：common/・bss.py・bss6.py・masks.py・masknet.py はこのノートブックの隣
import bss6                                              # noqa: E402
import common.plotting  # noqa: E402,F401（日本語フォント）
from common import metrics as M                          # noqa: E402
from common.stft import stft                             # noqa: E402
from common.audio_io import save                         # noqa: E402
from bss import demix                                    # noqa: E402

OUT = HERE / "out"
OUT.mkdir(exist_ok=True)
FS = 16000
WINS = ((512, 128), (2048, 512))


def evaluate(img, y, sdr_in, si_in):
    sdr, sir, sar, perm = M.bss_eval(img, y)
    si = M.si_sdr(img, y[perm])
    return dict(sdr=float(np.mean(sdr)), sir=float(np.mean(sir)), sar=float(np.mean(sar)),
                si_sdr=float(np.mean(si)), sdr_imp=float(np.mean(sdr - sdr_in)), si_imp=float(np.mean(si - si_in)))


def run_trial(src, rt60, seed, listen=None):
    mix, img = bss6.simulate(src, FS, rt60, seed)
    n = mix.shape[1]
    x0 = mix[0]
    ref_in = np.stack([x0, x0])
    sdr_in = M.bss_eval(img, ref_in, compute_permutation=False)[0]
    si_in = M.si_sdr(img, ref_in)
    rows, wavs = [], {"mix": x0}
    for L, S in WINS:
        X = stft(x0, L, S)
        Simg = np.stack([stft(img[k], L, S) for k in range(img.shape[0])])
        Mk = masks.ideal_masks(Simg, X)
        for nm, m in Mk.items():
            t0 = time.time()
            y = masks.apply_masks(m, X, L, S, n)
            rows.append(dict(method=f"{nm}", L=L, sec=time.time() - t0, **evaluate(img, y, sdr_in, si_in)))
            if L == 512:
                wavs[nm] = y[0]
    Xm = bss6.to_stft(mix)
    t0 = time.time()
    Y, _, _ = bss6.ilrma_v(Xm, 2, 100, seed=seed)          # 第7章の ilrma と同じ計算（IV-6）
    y = bss6.to_time(Y, n)
    rows.append(dict(method="ILRMA", L=2048, sec=time.time() - t0, **evaluate(img, y, sdr_in, si_in)))
    wavs["ILRMA"] = y[M.bss_eval(img, y)[3][0]]
    Sm = bss6.to_stft(img)
    y = bss6.to_time(demix(bss6.oracle_w(Xm, Sm), Xm), n)
    rows.append(dict(method="oracleW", L=2048, sec=0.0, **evaluate(img, y, sdr_in, si_in)))
    for r in rows:
        r.update(rt60=rt60, seed=seed, sdr_in=float(np.mean(sdr_in)))
    if listen is not None:                                # listen は wav の名前の先頭（パス）
        wavs["ref"] = img[0]
        for k, v in wavs.items():
            save(f"{listen}_{k}.wav", v / np.max(np.abs(v)) * 10 ** (-10 / 20), FS)
    return rows


def part0(src):
    print("[0] STFT の上でのマスクの誤差 Σ|s − m x|²/Σ|s|²（RT60 0.3 s・種 0、窓長 512、音源 2 つまとめて）")
    mix, img = bss6.simulate(src, FS, 0.3, 0)
    X = stft(mix[0], 512, 128)
    Simg = np.stack([stft(img[k], 512, 128) for k in range(2)])
    Mk = masks.ideal_masks(Simg, X)
    for nm, m in Mk.items():
        print(f"  {nm}：{masks.stft_error(m, Simg, X):7.2f} dB")
    psm_raw = np.real(Simg * np.conj(X)[None]) / (np.abs(X)[None] ** 2 + masks.EPS)
    print(f"  PSM（切らない）：{masks.stft_error(psm_raw, Simg, X):7.2f} dB、切らない PSM の範囲 "
          f"{psm_raw.min():.2f}〜{psm_raw.max():.2f}、[0,1] の外にあるビンの割合 {np.mean((psm_raw < 0) | (psm_raw > 1)):.3f}")
    rng = np.random.default_rng(0)
    worse = [masks.stft_error(np.clip(psm_raw + 0.05 * rng.standard_normal(psm_raw.shape), -5, 5), Simg, X)
             for _ in range(5)]
    print(f"  切らない PSM を乱数で少しずらすと：{np.round(worse, 2)} dB（どれも大きくなる＝最小の確かめ）")
    ibm = Mk["IBM"]
    flip = ibm.copy()
    idx = rng.random(ibm.shape) < 0.01
    flip[idx] = 1 - flip[idx]
    print(f"  IBM の 1% のビンを反転すると：{masks.stft_error(flip, Simg, X):7.2f} dB（IBM は 0/1 の中で最小）")
    irm2 = Mk["IRM"] ** 2
    print(f"  IRM の 2 乗と IWM の差の最大：{np.max(np.abs(irm2 - Mk['IWM'])):.1e}、"
          f"IRM の和 Σ_n m_n の平均 {Mk['IRM'].sum(axis=0).mean():.3f}（1 より大きい）、IWM の和 {Mk['IWM'].sum(axis=0).mean():.3f}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--rt60", type=float, nargs="+", default=[0.15, 0.3])
    ap.add_argument("--seeds", type=int, default=10)
    ap.add_argument("--out", default=str(OUT / "oracle_iv7.csv"))
    a = ap.parse_args()
    src = bss6.sources()
    part0(src)
    rows = []
    for rt in a.rt60:
        for seed in range(a.seeds):
            t0 = time.time()
            stem = str(pathlib.Path(a.out).with_suffix("")) + "_listen"
            rows += run_trial(src, rt, seed, listen=stem if (rt == 0.3 and seed == 0) else None)
            print(f"  RT60 {rt:.2f} s 種 {seed}（{time.time() - t0:.0f} 秒）", flush=True)
            pd.DataFrame(rows).to_csv(a.out, index=False)
    summarize(pd.DataFrame(rows), pathlib.Path(a.out).with_suffix(".png"))


def summarize(df, png):
    df = df.copy()
    df["name"] = df["method"] + np.where(df["method"].isin(["ILRMA", "oracleW"]), "（2ch）", "（窓長" + df["L"].astype(str) + "）")
    order = [f"{m}（窓長{L}）" for L, _ in WINS for m in masks.MASKS] + ["ILRMA（2ch）", "oracleW（2ch）"]
    n_seed = df.seed.nunique()
    print(f"\n[1] 平均 ± 標準偏差（種 {n_seed} 通り、2 音源の平均）。SDR・SI-SDR は改善量 [dB]")
    rts = sorted(df.rt60.unique())
    print("  方法             " + "".join(f"| RT60 {r:.2f} s: SDRi      SIR    SAR    SI-SDRi " for r in rts))
    for nm in order:
        g = df[df.name == nm]
        cells = ""
        for r in rts:
            h = g[g.rt60 == r]
            cells += (f"| {h.sdr_imp.mean():6.2f} ± {h.sdr_imp.std():4.2f} {h.sir.mean():6.1f} {h.sar.mean():6.1f} "
                      f"{h.si_imp.mean():6.2f} ± {h.si_imp.std():4.2f} ")
        print(f"  {nm:16s} {cells}")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(11, 4))
    w = 0.8 / len(rts)
    for j, r in enumerate(rts):
        mu = [df[(df.name == nm) & (df.rt60 == r)].sdr_imp.mean() for nm in order]
        sd = [df[(df.name == nm) & (df.rt60 == r)].sdr_imp.std() for nm in order]
        ax.bar(np.arange(len(order)) + (j - (len(rts) - 1) / 2) * w, mu, w, yerr=sd, capsize=3, label=f"RT60 {r * 1000:.0f} ms")
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=35, ha="right", fontsize=8)
    ax.set_ylabel("SDR 改善量 [dB]（平均 ± 標準偏差）")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(png, dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print(f"{png} を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"種 (\d+) 通り") == 1 and "out/oracle_try.png を保存しました" in cap.stdout, "--seeds 1 で動き、out/oracle_try.png ができた（本番とは別のファイル）")

本番（種 10 通り × 残響 2 条件 = 20 試行）です。教科書の目安は 10〜15 分（執筆時の Mac で、ほかの処理が無ければ 3 分ほど）。結果の `out/oracle_iv7.csv` は 5 節（手順 G）で使います。学習を使わないので、値は教科書と同じになります。

In [ ]:
%%capture cap
# ノートブック版：上のセルで定義した main() を、ターミナルで引数を変えて実行するのと同じように呼ぶ
import sys
for argv in [['oracle_masks.py']]:
    sys.argv = argv
    print('$ python', ' '.join(argv), flush=True)
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
err = {k: num(rf"(?m)^  {k}：\s*([-\d.]+) dB") for k in ("IBM", "IRM", "IWM", "IAM", "PSM", "cRM")}
want = dict(IBM=-10.6, IRM=-11.4, IAM=-11.5, IWM=-12.4, PSM=-13.2, cRM=-20.7)
ok(all(abs(err[k] - v) < 0.1 for k, v in want.items()), f"[0] 誤差が IBM −10.6、IRM −11.4、IAM −11.5、IWM −12.4、PSM −13.2、cRM −20.7 dB {err}")
ok(abs(num(r"PSM（切らない）：\s*([-\d.]+) dB") + 14.2) < 0.1, "[0] 切らない PSM は −14.2 dB（切った PSM より小さい）")
def cell(name, i):
    m = re.search(rf"^  {re.escape(name)}\s*\|\s*([-\d.]+) ±.*?\|\s*([-\d.]+) ±", cap.stdout, re.M)
    return float(m.group(i))
for name, v in (("PSM（窓長512）", (14.47, 14.24)), ("ILRMA（2ch）", (16.76, 5.66)), ("oracleW（2ch）", (20.32, 10.91))):
    got = (cell(name, 1), cell(name, 2))
    ok(all(abs(a - b) < 0.1 for a, b in zip(got, v)), f"[1] {name} が {v[0]}・{v[1]} dB（{got[0]}・{got[1]}。ILRMA・oracleW は IV-6 の表と同じ）")
ok(num(r"種 (\d+) 通り") == 10, "[1] 種 10 通り × 2 条件の本番")
import os
ok(all(os.path.exists(f"out/oracle_iv7_listen_{k}.wav") for k in ("ref", "mix", "IBM", "IRM", "IWM", "IAM", "PSM", "cRM", "ILRMA")),
   "out/oracle_iv7.csv と out/oracle_iv7_listen_*.wav ができた")

聞き比べます（手順 D の 2、音量は小さく）：正解の像 → 混合音 → IBM → IWM → PSM → cRM → ILRMA。IBM にピコピコ・チリチリとしたミュージカルノイズ、IWM・PSM はそれが少ない、cRM はほぼ正解と同じ、ILRMA は妨害の声が少し残るがゆがみが少ない、と聞こえるでしょうか。SDR の順と、聞いた感じの順が同じか違うかをノートに書いてください。wav は 1 つずつ最大値をそろえているので、ファイルどうしの音量の差には意味がありません。

In [ ]:
# 保存した wav を聞く（音量を小さくしてから。1つずつ）
for f in ['out/oracle_iv7_listen_ref.wav', 'out/oracle_iv7_listen_mix.wav', 'out/oracle_iv7_listen_IBM.wav', 'out/oracle_iv7_listen_IWM.wav', 'out/oracle_iv7_listen_PSM.wav', 'out/oracle_iv7_listen_cRM.wav', 'out/oracle_iv7_listen_ILRMA.wav']:
    print(f)
    listen_file(f)

## 3. 学習の準備（IV-7-3-9・3-10、手順 E、`sepdata.py`・`masknet.py`）

学習用の部屋（学習 300・検証 40）と音声の区間を作ります（1〜3 分）。テストの区間（libri1・libri3 の最初の 8 秒）と重ならない区間だけを使います。2 回目の `--leak` は、わざとテストの区間も学習に入れたもの（データ漏れの実験用。`out/speech_leak.npz`）です。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['sepdata.py']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-7：小さなマスク推定ネットワークのための、学習用の部屋と音声を用意する

使い方: cd ~/home-lab/partIV/iv7 && uv run python sepdata.py [--rooms 300] [--leak]
  音声は librosa の例の音源 libri1・libri2・libri3（LibriSpeech の抜粋、話者 3 人）。
  評価（テスト）に使う区間は IV-6 と同じ libri1・libri3 の最初の 8 秒。学習と検証には、それと重ならない区間だけを使う。
      libri1：テスト [0, 8) 秒、学習 [8, 末尾−2)、検証 [末尾−2, 末尾)
      libri2：学習 [0, 末尾−2)、検証 [末尾−2, 末尾)
      libri3：テスト [0, 8)、学習 [8, 末尾−2)、検証 [末尾−2, 末尾)
  --leak を付けると、わざとテストの区間も学習に入れる（データ漏れの実験。out/speech_leak.npz）。
  部屋は pyroomacoustics の鏡像法で、IV-6 の bss6.simulate と同じ形（幅 5〜8 m、奥行 4〜6 m、高さ 3 m、
  間隔 4 cm の 2 本のマイクの 0 番の位置）。ただしマイクの高さは 1.2 m（音源は 1.5 m。MIC_Z の注）。RT60 は 0.15〜0.45 秒の一様乱数（0.13 秒より短いと、大きな部屋では吸音率が 1 を超えて作れない）、音源の距離は 0.8〜1.5 m。
  学習の部屋は種 1000 から、検証の部屋は種 5000 から（テストの部屋は種 0〜9。どれも重ならない）。
出力：out/rooms_train.npz・out/rooms_valid.npz（インパルス応答 (部屋, 2 音源, 長さ)）、out/speech.npz
計算時間の目安：300 部屋で 1〜3 分
"""
import argparse
import pathlib

import numpy as np

HERE = pathlib.Path(".").resolve()   # ノートブック版：このノートブックのフォルダ（out/ と runs/ もここ）
OUT = HERE / "out"
OUT.mkdir(exist_ok=True)
FS = 16000
TEST_SEC = 8.0
MIC_Z = 1.2          # マイクの高さ [m]。音源（1.5 m）と同じ高さ＝天井 3 m の半分にすると、床と天井の鏡像が
                     # 同じ時刻に重なり、直接音と残響の比（DRR）が約 1.7 dB 低く出る（IV-3-3-6）。そこで 1.2 m にした
VALID_SEC = 2.0


def load_speakers(fs=FS):
    import librosa
    sp = {}
    for nm in ("libri1", "libri2", "libri3"):
        x, _ = librosa.load(librosa.ex(nm), sr=fs, mono=True)
        sp[nm] = x / (np.std(x) + 1e-12)
    return sp


def split_speech(sp, leak=False):
    """話者ごとに学習・検証の区間を切り出す。戻り値 {"train": {名前: 配列}, "valid": {...}}"""
    t, v = {}, {}
    nt, nv = int(TEST_SEC * FS), int(VALID_SEC * FS)
    for nm, x in sp.items():
        start = 0 if (nm == "libri2" or leak) else nt
        t[nm] = x[start:len(x) - nv]
        v[nm] = x[len(x) - nv:]
    return {"train": t, "valid": v}


def make_rooms(n, seed0, fs=FS, rt_range=(0.15, 0.45), d=0.04, min_sep_deg=30):
    """n 個の部屋のインパルス応答 (n, 2, 長さ) と RT60 を返す（マイク 0 番だけ）"""
    import pyroomacoustics as pra
    rirs, rts = [], []
    for i in range(n):
        rng = np.random.default_rng(seed0 + i)
        dim = np.array([rng.uniform(5, 8), rng.uniform(4, 6), 3.0])
        rt = rng.uniform(*rt_range)
        e_abs, max_order = pra.inverse_sabine(rt, dim)
        room = pra.ShoeBox(dim, fs=fs, materials=pra.Material(e_abs), max_order=max_order)
        c = np.array([dim[0] / 2 + rng.uniform(-0.5, 0.5), dim[1] / 2 + rng.uniform(-0.5, 0.5), 1.5])
        room.add_microphone_array(np.array([c[0] - d / 2, c[1], MIC_Z])[:, None])   # マイクだけ高さ 1.2 m（上の MIC_Z の注）
        while True:
            ang = np.sort(rng.uniform(0, np.pi, 2))
            if np.diff(ang)[0] >= np.radians(min_sep_deg):
                break
        for k in range(2):
            dist = rng.uniform(0.8, 1.5)
            room.add_source(c + dist * np.array([np.cos(ang[k]), np.sin(ang[k]), 0.0]))
        room.compute_rir()
        rirs.append([np.asarray(room.rir[0][k]) for k in range(2)])
        rts.append(rt)
    Lmax = max(len(h) for pair in rirs for h in pair)
    R = np.zeros((n, 2, Lmax), dtype=np.float32)
    for i, pair in enumerate(rirs):
        for k, h in enumerate(pair):
            R[i, k, :len(h)] = h
    return R, np.array(rts)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--rooms", type=int, default=300)
    ap.add_argument("--leak", action="store_true")
    a = ap.parse_args()
    sp = load_speakers()
    sets = split_speech(sp, leak=a.leak)
    name = "speech_leak.npz" if a.leak else "speech.npz"
    np.savez(OUT / name, **{f"{k}_{nm}": x.astype(np.float32) for k, d in sets.items() for nm, x in d.items()})
    for k, d in sets.items():
        print(f"{k}: " + "、".join(f"{nm} {len(x) / FS:.2f} 秒" for nm, x in d.items()))
    print(f"out/{name} を保存しました")
    if a.leak:
        return
    for tag, n, s0 in (("train", a.rooms, 1000), ("valid", 40, 5000)):
        R, rts = make_rooms(n, s0)
        np.savez(OUT / f"rooms_{tag}.npz", rir=R, rt60=rts)
        print(f"rooms_{tag}: {n} 部屋、RT60 {rts.min():.2f}〜{rts.max():.2f} 秒、インパルス応答の長さ {R.shape[2]} 標本")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
t = {nm: num(rf"train: .*?{nm} ([\d.]+) 秒") for nm in ("libri1", "libri2", "libri3")}
ok(t == dict(libri1=4.84, libri2=14.74, libri3=3.91), f"学習の区間が libri1 4.84 秒・libri2 14.74 秒・libri3 3.91 秒 {t}")
ok(re.search(r"valid: libri1 2\.00 秒、libri2 2\.00 秒、libri3 2\.00 秒", cap.stdout) is not None, "検証が各 2.00 秒")
ok(num(r"rooms_train: (\d+) 部屋") == 300 and num(r"rooms_valid: (\d+) 部屋") == 40, "部屋は学習 300・検証 40")
lo, hi = num(r"rooms_train: \d+ 部屋、RT60 ([\d.]+)〜"), num(r"rooms_train: \d+ 部屋、RT60 [\d.]+〜([\d.]+) 秒")
ok(0.15 <= lo and hi <= 0.45, f"RT60 は 0.15〜0.45 秒（{lo}〜{hi}）")

In [ ]:
%%capture cap
# ノートブック版：上のセルで定義した main() を、ターミナルで引数を変えて実行するのと同じように呼ぶ
import sys
for argv in [['sepdata.py', '--leak']]:
    sys.argv = argv
    print('$ python', ' '.join(argv), flush=True)
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"train: libri1 ([\d.]+) 秒") == 12.84 and num(r"train: .*?libri3 ([\d.]+) 秒") == 11.91 and "out/speech_leak.npz を保存しました" in cap.stdout,
   "--leak では学習の区間が libri1 12.84 秒・libri3 11.91 秒（テストの 8 秒も入る）")

`masknet.py` の確かめ（教科書では `uv run python masknet.py`）：torch の STFT と common の STFT、PIT の行列の計算と素朴な計算、fast_bss_eval の `si_sdr_pit_loss` を照合し、ネットワークの大きさと計算の場所（device）を表示します。

In [ ]:
%%capture cap
# ノートブック版：教科書の `uv run python masknet.py`（中で self_check() を呼ぶ）と同じ
import masknet
masknet.self_check()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"\[1\] .*?差の最大 ([\d.e+-]+)（float64）") < 1e-12, "[1] torch と common の STFT の差が 1e-14 程度")
ok(num(r"最初のフレームの差 ([\d.]+)") > 1, f"[1] pad_mode を既定（reflect）のままにすると最初のフレームで大きくずれる（{num(r'最初のフレームの差 ([\d.]+)')}。教科書 約 24）")
ok(abs(num(r"行列で ([\d.]+)、") - num(r"素朴に ([\d.]+)、")) < 1e-6, "[2] PIT の行列と素朴な計算の差が 1e-9 程度（許容 1e-6）")
ok(num(r"正解の順番を入れ替えても同じ：([\d.]+)") == num(r"行列で ([\d.]+)、"), "[2] 正解の順番を入れ替えても PIT の損失は同じ")
if "[3] fast_bss_eval が無い" not in cap.stdout:
    ok(abs(num(r"自前 ([-\d.]+)、fast_bss_eval") - num(r"si_sdr_pit_loss ([-\d.]+)")) < 1e-8 and "[1, 0]" in cap.stdout,
       "[3] fast_bss_eval の si_sdr_pit_loss と一致し、入れ替えた正解を見つけた")
dev = re.search(r"使える device：(\w+)", cap.stdout).group(1)
ok("パラメータ 924,164 個" in cap.stdout and dev in ("cuda", "mps", "cpu"), f"[4] パラメータ 924,164 個、device が {dev}（Mac では mps、Colab の GPU では cuda）")

## 4. 学習して評価する（IV-7-3-9・3-12、手順 F、`train_masknet.py`・`eval_masknet.py`）

**ノートブックの既定は試しの設定**（`--steps 300 --eval-every 100`）です。教科書の本番（2000 回、250 回ごと）にするときは、下の 2 つのセルの `"--steps", "300", "--eval-every", "100"` を消してください（1 回 5〜12 分。Colab の GPU では速く、CPU では数倍遅くなります）。

まず学習の種 0 で 1 回学習し、経過を見ます。100 回ごと（本番は 250 回ごと）に「学習の損失・検証の損失・検証の SI-SDR」が表示されます。本番では、学習の損失が 0.11 → 0.05 前後まで下がり続ける一方、検証の SI-SDR は 1〜2.5 dB の間で上下しながら頭打ちになります（過学習）。試しの設定では、まだ頭打ちの手前で終わります。

注意：前の実行の ckpt が残っていると、あとの評価（`--all-ckpt`）がそれも数えてしまうので、このセルは最初に `runs/` を消します（ノートブック版で足した 2 行）。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import shutil
shutil.rmtree("runs", ignore_errors=True)   # ノートブック版：前の実行の ckpt を消す
# 本番（教科書の手順 F の 1）は、下の sys.argv から "--steps", "300", "--eval-every", "100" を消す
import sys
sys.argv = ['train_masknet.py', '--seed', '0', '--out', 'runs/irm_s0', '--steps', '300', '--eval-every', '100']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-7：小さなマスク推定ネットワーク（BLSTM）を、PIT の損失で学習する

使い方: cd ~/home-lab/partIV/iv7 && uv run python train_masknet.py --seed 0 --out runs/irm_s0
  先に sepdata.py で out/speech.npz・out/rooms_train.npz・out/rooms_valid.npz を作っておく。
  --loss irm（既定）：IRM を目標にしたマスクの 2 乗誤差（PIT）。--loss psa：位相を考えた信号の近似（PIT）
  --speech out/speech_leak.npz：テストの区間も学習に入れる（データ漏れの実験。評価には使わない）
  種 --seed は、モデルの初期値（CPU で作ってから device に移す）と、学習の混合音の作り方の両方を決める。
  検証の混合音（64 個）は種によらず同じ（検証の部屋・検証の区間だけで作る）。
  --eval-every ごとに検証の損失と SI-SDR を測り、ckpt_<step>.pt を保存する。検証の SI-SDR が最大のものを best.pt にする
  （損失の種類によらず同じ物差しで選ぶため。検証の損失が最小の回とは一致しないことがある。テストは見ない）。
出力：<out>/metrics.csv（step,train_loss,valid_loss,valid_si_sdr,sec）、ckpt_*.pt、best.pt、run.json
計算時間の目安：既定（2000 回、バッチ 16、2 秒）で MPS 5〜12 分（Apple シリコン の MacBook Air、執筆時。ほかの作業で延びる）
"""
import argparse
import json
import pathlib
import time

import numpy as np
import torch

import masknet as mn

HERE = pathlib.Path(".").resolve()   # ノートブック版：このノートブックのフォルダ（out/ と runs/ もここ）


def batch_to_torch(mix, img, dev):
    mix = torch.from_numpy(mix).to(dev)
    img = torch.from_numpy(img).to(dev)
    B, N, n = img.shape
    X = mn.torch_stft(mix)                                       # (B, F, T)
    S = mn.torch_stft(img.reshape(B * N, n)).reshape(B, N, *X.shape[1:])
    return X, S


def loss_fn(model, X, S, kind):
    m = model(mn.features(X))                                    # (B, N, F, T)
    if kind == "irm":
        return mn.pit_loss(m, mn.irm_target(S))[0]
    pair = mn.psa_pair_factory(X.abs(), torch.angle(X))
    return mn.pit_loss(m, S, pair)[0]


def valid_si_sdr(model, X, img):
    """検証の混合音を分け（マスク × 混合音 → 逆 STFT）、PIT で対応づけた SI-SDR [dB] の平均を返す（損失の種類によらない物差し）"""
    m = model(mn.features(X))                                    # (B, N, F, T)
    B, N, n = img.shape
    w = torch.hann_window(mn.L_WIN, periodic=True, device=X.device)
    y = torch.istft((m * X[:, None]).reshape(B * N, *X.shape[1:]), n_fft=mn.L_WIN, hop_length=mn.S_HOP,
                    window=w, center=True, length=n).reshape(B, N, n)

    def neg_si(a, b):
        al = (a * b).sum(-1) / ((b * b).sum(-1) + 1e-8)
        t = al[..., None] * b
        return -10 * torch.log10((t ** 2).sum(-1) / (((t - a) ** 2).sum(-1) + 1e-8) + 1e-8)
    return -float(mn.pit_loss(y, img, neg_si)[0])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", required=True)
    ap.add_argument("--steps", type=int, default=2000)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--seg", type=float, default=2.0)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--hidden", type=int, default=128)
    ap.add_argument("--loss", choices=["irm", "psa"], default="irm")
    ap.add_argument("--dropout", type=float, default=0.3)
    ap.add_argument("--speed", type=int, default=1, help="1：速さを変えるデータ拡張をする（既定）、0：しない")
    ap.add_argument("--speech", default=str(HERE / "out" / "speech.npz"))
    ap.add_argument("--eval-every", type=int, default=250)
    ap.add_argument("--device", default="auto")
    a = ap.parse_args()
    out = pathlib.Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    dev = mn.pick_device(a.device)
    torch.manual_seed(a.seed)
    model = mn.MaskNet(hidden=a.hidden, dropout=a.dropout)                          # 初期値は CPU で作る（III-4 と同じ）
    model.to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=a.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=a.steps)
    train = mn.Mixer(a.speech, HERE / "out" / "rooms_train.npz", "train", a.seg, np.random.default_rng(1000 + a.seed),
                     speed=bool(a.speed))
    valid = mn.Mixer(HERE / "out" / "speech.npz", HERE / "out" / "rooms_valid.npz", "valid", a.seg,
                     np.random.default_rng(12345))
    vmix, vimg = valid.make(64)
    vX, vS = batch_to_torch(vmix, vimg, dev)
    vimg = torch.from_numpy(vimg).to(dev)
    print(f"device {dev}、パラメータ {mn.n_params(model):,} 個、学習の話者 {train.names}、損失 {a.loss}", flush=True)
    rec = dict(vars(a), device=str(dev), torch=torch.__version__, n_params=mn.n_params(model))
    rows, run_loss, best, t0 = [], [], -np.inf, time.time()
    for step in range(1, a.steps + 1):
        model.train()
        X, S = batch_to_torch(*train.make(a.batch), dev)
        loss = loss_fn(model, X, S, a.loss)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        sched.step()
        run_loss.append(loss.item())
        if step % a.eval_every == 0 or step == a.steps:
            model.eval()
            with torch.no_grad():
                vl = float(loss_fn(model, vX, vS, a.loss))
                vsi = valid_si_sdr(model, vX, vimg)
            rows.append(dict(step=step, train_loss=float(np.mean(run_loss)), valid_loss=vl, valid_si_sdr=vsi,
                             sec=time.time() - t0))
            run_loss = []
            state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            torch.save(dict(state=state, hidden=a.hidden, dropout=a.dropout, step=step), out / f"ckpt_{step}.pt")
            if vsi > best:
                best = vsi
                torch.save(dict(state=state, hidden=a.hidden, dropout=a.dropout, step=step), out / "best.pt")
            print(f"  {step:5d} 回  学習 {rows[-1]['train_loss']:.4f}  検証 {vl:.4f}（SI-SDR {vsi:5.2f} dB）  "
                  f"{rows[-1]['sec']:.0f} 秒", flush=True)
            import pandas as pd
            pd.DataFrame(rows).to_csv(out / "metrics.csv", index=False)
    rec.update(best_valid_si_sdr=best, best_step=int(max(rows, key=lambda r: r["valid_si_sdr"])["step"]),
               min_loss_step=int(min(rows, key=lambda r: r["valid_loss"])["step"]), sec=time.time() - t0)
    (out / "run.json").write_text(json.dumps(rec, ensure_ascii=False, indent=1))
    print(f"終わり：検証の SI-SDR が最大 {best:.2f} dB（{rec['best_step']} 回。損失が最小の回は {rec['min_loss_step']} 回）、"
          f"{rec['sec'] / 60:.1f} 分")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
rows = [tuple(map(float, m)) for m in re.findall(r"^\s+(\d+) 回  学習 ([\d.]+)  検証 ([\d.]+)（SI-SDR\s+([-\d.]+) dB）", cap.stdout, re.M)]
import json, os, pandas as pd
rec = json.load(open("runs/irm_s0/run.json"))
n_eval = -(-rec["steps"] // rec["eval_every"])
ok(len(rows) == n_eval and rows[-1][0] == rec["steps"], f"{rec['eval_every']} 回ごとに学習の損失・検証の損失・検証の SI-SDR が表示された（{len(rows)} 行）")
ok(rows[-1][1] < rows[0][1], f"学習の損失が下がっている（{rows[0][1]} → {rows[-1][1]}。本番では 0.11 → 0.05 前後）")
ok("検証の SI-SDR が最大" in cap.stdout and "損失が最小の回は" in cap.stdout,
   f"終わりの行に「検証の SI-SDR が最大の回」（{rec['best_step']} 回）と「検証の損失が最小の回」（{rec['min_loss_step']} 回）が出た（2 つは違うことが多い）")
ok(os.path.exists("runs/irm_s0/best.pt") and len(pd.read_csv("runs/irm_s0/metrics.csv")) == n_eval
   and abs(rec["best_valid_si_sdr"] - max(r[3] for r in rows)) < 0.01, "best.pt（検証の SI-SDR が最大の回）と metrics.csv ができた")
print(f"学習に使った device：{rec['device']}（この計算機で選ばれる device：{masknet.pick_device()}）")

残りの種と、漏れありの学習、PSA の損失での学習を回します（教科書の `run_iv7.sh` の F の行。試しの設定で 8 回、執筆時の Mac で約 5 分）。途中の表示は、終わってから次のセルでまとめて見ます。

In [ ]:
%%capture cap
# ノートブック版：上のセルで定義した main() を、ターミナルで引数を変えて実行するのと同じように呼ぶ
# 本番は、各行の "--steps", "300", "--eval-every", "100" を消す
import sys
for argv in [['train_masknet.py', '--seed', '1', '--out', 'runs/irm_s1', '--steps', '300', '--eval-every', '100'],
             ['train_masknet.py', '--seed', '2', '--out', 'runs/irm_s2', '--steps', '300', '--eval-every', '100'],
             ['train_masknet.py', '--seed', '0', '--out', 'runs/leak_s0', '--speech', 'out/speech_leak.npz', '--steps', '300', '--eval-every', '100'],
             ['train_masknet.py', '--seed', '1', '--out', 'runs/leak_s1', '--speech', 'out/speech_leak.npz', '--steps', '300', '--eval-every', '100'],
             ['train_masknet.py', '--seed', '2', '--out', 'runs/leak_s2', '--speech', 'out/speech_leak.npz', '--steps', '300', '--eval-every', '100'],
             ['train_masknet.py', '--seed', '0', '--out', 'runs/psa_s0', '--loss', 'psa', '--steps', '300', '--eval-every', '100'],
             ['train_masknet.py', '--seed', '1', '--out', 'runs/psa_s1', '--loss', 'psa', '--steps', '300', '--eval-every', '100'],
             ['train_masknet.py', '--seed', '2', '--out', 'runs/psa_s2', '--loss', 'psa', '--steps', '300', '--eval-every', '100']]:
    sys.argv = argv
    print('$ python', ' '.join(argv), flush=True)
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
import json, os, glob
names = [f"{t}_s{s}" for t in ("irm", "leak", "psa") for s in (0, 1, 2)]
recs = {n: json.load(open(f"runs/{n}/run.json")) for n in names}
ok(all(os.path.exists(f"runs/{n}/best.pt") for n in names), "9 回の学習（種 3 つ × IRM・漏れあり・PSA）がすべて終わり、best.pt ができた")
ok(all(r["best_step"] % r["eval_every"] == 0 or r["best_step"] == r["steps"] for r in recs.values())
   and {r["loss"] for n, r in recs.items() if n.startswith("psa")} == {"psa"} and recs["leak_s0"]["speech"].endswith("speech_leak.npz"),
   "best.pt は途中の検証の回のどれか。PSA は --loss psa、漏れありは out/speech_leak.npz で学習した")
print("検証の SI-SDR の最大（試しの設定。本番では IRM 2.15〜2.53 dB、PSA 2.46〜2.98 dB）：",
      {n: round(r["best_valid_si_sdr"], 2) for n, r in recs.items()})

全部を、IV-6 と同じテストの部屋（種 0〜9 × 残響 2 条件）で評価します（`--all-ckpt`：途中の ckpt もすべて。5 節の誤用 (b) で使う）。数分かかります。`eval_masknet.py` は前の結果の CSV に行を足していくので、ノートブック版では最初に `out/eval_masknet.csv` を消します。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import os
if os.path.exists("out/eval_masknet.csv"):
    os.remove("out/eval_masknet.csv")   # ノートブック版：前の実行の行を残さない
import sys
sys.argv = ['eval_masknet.py', 'runs/irm_s0', 'runs/irm_s1', 'runs/irm_s2', 'runs/leak_s0', 'runs/leak_s1', 'runs/leak_s2', 'runs/psa_s0', 'runs/psa_s1', 'runs/psa_s2', '--all-ckpt']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-7：学習したマスク推定ネットワークを、IV-6 と同じテストの部屋（種 0〜9 × 残響 2 条件）で評価する

使い方: cd ~/home-lab/partIV/iv7 && uv run python eval_masknet.py runs/irm_s0 runs/irm_s1 … [--all-ckpt]
  テストの混合音は oracle_masks.py・IV-6 と同じ（bss6.simulate、libri1・libri3 の最初の 8 秒）。
  ネットワークはマイク 0 番の混合音だけを使う（1 チャネル）。マスクは混合音の STFT（L = 512、S = 128）に掛ける。
  既定は best.pt（検証の SI-SDR が最大の回）だけ。--all-ckpt で途中の ckpt_*.pt もすべて評価する（統計の誤用の実験用）。
出力：out/eval_masknet.csv（列 run,tag,tseed,ckpt,step,rt60,seed,sdr_in,sdr,sir,sar,si_sdr,sdr_imp,si_imp）、
      out/listen_net.wav（最初の run・RT60 0.3 s・種 0、最大 −10 dBFS）
計算時間の目安：run 1 つ（best.pt だけ）で 1〜2 分
"""
import argparse
import pathlib
import re
import sys

import numpy as np
import pandas as pd
import torch

import masknet as mn
import masks
HERE = pathlib.Path(".").resolve()   # ノートブック版：このノートブックのフォルダ（out/ と runs/ もここ）
sys.path.insert(0, ".")   # ノートブック版：common/・bss.py・bss6.py・masks.py・masknet.py はこのノートブックの隣
import bss6                                              # noqa: E402
from common import metrics as M                          # noqa: E402
from common.stft import stft                             # noqa: E402
from common.audio_io import save                         # noqa: E402

OUT = HERE / "out"
FS = 16000


def load_model(path, dev):
    ck = torch.load(path, map_location="cpu")
    model = mn.MaskNet(hidden=ck["hidden"])
    model.load_state_dict(ck["state"])
    return model.to(dev).eval(), ck["step"]


def separate(model, x0, dev):
    """1 チャネルの混合音 x0 (n,) → 分離音 (2, n)"""
    X = stft(x0, mn.L_WIN, mn.S_HOP)                          # (F, T) numpy（common.stft）
    xt = torch.from_numpy((x0 / (np.std(x0) + 1e-8)).astype(np.float32))[None].to(dev)
    with torch.no_grad():
        m = model(mn.features(mn.torch_stft(xt)))[0].cpu().numpy().astype(float)   # (2, F, T)
    return masks.apply_masks(m, X, mn.L_WIN, mn.S_HOP, len(x0))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("runs", nargs="+")
    ap.add_argument("--rt60", type=float, nargs="+", default=[0.15, 0.3])
    ap.add_argument("--seeds", type=int, default=10)
    ap.add_argument("--all-ckpt", action="store_true")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--out", default=str(OUT / "eval_masknet.csv"))
    a = ap.parse_args()
    dev = mn.pick_device(a.device)
    src = bss6.sources()
    tests = {}
    for rt in a.rt60:
        for seed in range(a.seeds):
            mix, img = bss6.simulate(src, FS, rt, seed)
            sdr_in = M.bss_eval(img, np.stack([mix[0], mix[0]]), compute_permutation=False)[0]
            si_in = M.si_sdr(img, np.stack([mix[0], mix[0]]))
            tests[(rt, seed)] = (mix[0], img, sdr_in, si_in)
    rows = []
    for k, run in enumerate(a.runs):
        run = pathlib.Path(run)
        tag, tseed = re.match(r"(.+)_s(\d+)$", run.name).groups()
        cks = [run / "best.pt"]
        if a.all_ckpt:
            cks += sorted(run.glob("ckpt_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
        for ck in cks:
            model, step = load_model(ck, dev)
            for (rt, seed), (x0, img, sdr_in, si_in) in tests.items():
                y = separate(model, x0, dev)
                sdr, sir, sar, perm = M.bss_eval(img, y)
                si = M.si_sdr(img, y[perm])
                rows.append(dict(run=run.name, tag=tag, tseed=int(tseed), ckpt=ck.stem, step=step, rt60=rt, seed=seed,
                                 sdr_in=float(np.mean(sdr_in)), sdr=float(np.mean(sdr)), sir=float(np.mean(sir)),
                                 sar=float(np.mean(sar)), si_sdr=float(np.mean(si)),
                                 sdr_imp=float(np.mean(sdr - sdr_in)), si_imp=float(np.mean(si - si_in))))
                if k == 0 and ck.stem == "best" and rt == 0.3 and seed == 0:
                    v = y[perm[0]]
                    save(OUT / "listen_net.wav", v / np.max(np.abs(v)) * 10 ** (-10 / 20), FS)
            g = pd.DataFrame([r for r in rows if r["run"] == run.name and r["ckpt"] == ck.stem])
            print(f"  {run.name} {ck.stem}（{step} 回）：SDR 改善量 " +
                  "、".join(f"RT60 {r:.2f} s {h.sdr_imp.mean():5.2f} ± {h.sdr_imp.std():4.2f}" for r, h in g.groupby("rt60")),
                  flush=True)
    df = pd.DataFrame(rows)
    old = pathlib.Path(a.out)
    if old.exists():                                       # 同じ run・ckpt の行は置き換え、ほかは残す
        prev = pd.read_csv(old)
        keep = ~prev.set_index(["run", "ckpt"]).index.isin(df.set_index(["run", "ckpt"]).index)
        df = pd.concat([prev[keep], df], ignore_index=True)
    df.to_csv(old, index=False)
    print(f"{old} に {len(df)} 行を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
import glob, os, pandas as pd
df = pd.read_csv("out/eval_masknet.csv")
n_ck = len(glob.glob("runs/irm_s0/ckpt_*.pt"))
ok(len(df) == 9 * (1 + n_ck) * 20, f"out/eval_masknet.csv が 9 回 ×（best ＋ 途中の {n_ck} 個）× 20 試行 = {len(df)} 行（本番では 1620 行）")
ok(os.path.exists("out/listen_net.wav"), "out/listen_net.wav ができた")
best = df[df.ckpt == "best"]
irm = best[best.tag == "irm"].groupby("rt60").sdr_imp.mean()
ok(all(irm < 11), f"ネットワーク（IRM、best）の SDR 改善量は理想比マスク（約 12 dB）に届かない {irm.round(2).to_dict()}（本番では 3 dB 前後）")

ネットワークの出力を聞きます（手順 F の 3、音量は小さく）：`out/listen_net.wav`（種 0、RT60 0.3 s・部屋 0）と、`out/oracle_iv7_listen_IRM.wav`（理想比マスク）。妨害の声がどれだけ残っているか、声がこもっていないかをノートに書いてください。理想マスクとの差が「推定の誤り（データの少なさ）」による分です（試しの設定では、学習が短い分だけ本番より妨害の声が多く残ります）。

In [ ]:
# 保存した wav を聞く（音量を小さくしてから。1つずつ）
for f in ['out/listen_net.wav', 'out/oracle_iv7_listen_IRM.wav']:
    print(f)
    listen_file(f)

## 5. 統計で比べ、誤用を確かめる（IV-7-3-11・3-12、手順 G、`stats_iv7.py`）

2 節の `out/oracle_iv7.csv` と 4 節の `out/eval_masknet.csv` を使います（30 秒ほど）。試しの設定では、ネットワークの行（Net・Net（PSA）・Net（漏れあり））と、[2] の Net の差、[4]〜[6] の値が教科書（3-12）と合いません。理想マスク・ILRMA・oracleW の行と [5] の (d) は教科書と同じ値になります。[5] の (a)〜(d) の数値をノートに写し、どれが「楽観的なかたより」かを書いてください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['stats_iv7.py']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
#!/usr/bin/env python3
"""IV-7：ネットワーク・ILRMA・理想マスクの比較を、部屋（種）を単位に統計で行い、統計の誤用を数値で確かめる

使い方: cd ~/home-lab/partIV/iv7 && uv run python stats_iv7.py
  入力：out/oracle_iv7.csv（oracle_masks.py）、out/eval_masknet.csv（eval_masknet.py。irm_s0〜2・leak_s0〜2・psa_s0〜2、--all-ckpt）
  [1] 方法ごとの SDR 改善量（平均 ± 標準偏差。単位は部屋。ネットワークは学習の種 3 つの平均を部屋ごとにとる）
  [2] 対にした比較（同じ部屋どうしの差）：t 区間・ウィルコクソン、4 つの検定をホルム法で補正
  [3] 線形混合モデル：SDR 改善量 ~ 方法 × 残響、部屋（種）を変量効果
  [4] ばらつきの内訳：部屋による差と、学習の種による差（交差した変量効果）
  [5] 統計の誤用：(a) 学習の種 1 つで比べる (b) テストの成績で途中の回を選ぶ (c) データ漏れ (d) 平均だけで比べる
  [6] 損失の比較（PSA − IRM）を学習の種で対にする（III-4 と同じ考え方）
計算時間の目安：30 秒
"""
import pathlib
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

HERE = pathlib.Path(".").resolve()   # ノートブック版：このノートブックのフォルダ（out/ と runs/ もここ）
OUT = HERE / "out"


def holm(p):
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    run = 0.0
    for i, j in enumerate(order):
        run = max(run, (len(p) - i) * p[j])
        adj[j] = min(1.0, run)
    return adj


def paired(d):
    d = np.asarray(d)
    ci = stats.t.interval(0.95, len(d) - 1, loc=d.mean(), scale=d.std(ddof=1) / np.sqrt(len(d)))
    return d.mean(), ci, stats.wilcoxon(d).pvalue, int(np.sum(d > 0))


def load():
    orc = pd.read_csv(OUT / "oracle_iv7.csv")
    net = pd.read_csv(OUT / "eval_masknet.csv")
    o = orc[((orc.method.isin(["IRM", "PSM"])) & (orc.L == 512)) | orc.method.isin(["ILRMA", "oracleW"])].copy()
    o["method"] = o["method"].replace({"IRM": "IRM（oracle）", "PSM": "PSM（oracle）"})
    best = net[net.ckpt == "best"]
    nets = (best[best.tag == "irm"].groupby(["rt60", "seed"])["sdr_imp"].mean().reset_index().assign(method="Net"))
    leak = (best[best.tag == "leak"].groupby(["rt60", "seed"])["sdr_imp"].mean().reset_index().assign(method="Net（漏れあり）"))
    psa = (best[best.tag == "psa"].groupby(["rt60", "seed"])["sdr_imp"].mean().reset_index().assign(method="Net（PSA）"))
    wide = pd.concat([o[["rt60", "seed", "method", "sdr_imp"]], nets, psa, leak], ignore_index=True)
    return wide, net


def main():
    wide, net = load()
    order = ["Net", "Net（PSA）", "ILRMA", "IRM（oracle）", "PSM（oracle）", "oracleW", "Net（漏れあり）"]
    print("[1] SDR 改善量 [dB]：平均 ± 標準偏差（部屋 10 通り。Net は学習の種 3 つの平均を部屋ごとにとったもの）")
    for m in order:
        g = wide[wide.method == m]
        print(f"  {m:14s}" + "".join(f"  RT60 {r:.2f} s：{h.sdr_imp.mean():6.2f} ± {h.sdr_imp.std():4.2f}"
                                      for r, h in g.groupby("rt60")))
    piv = wide.pivot_table(index=["rt60", "seed"], columns="method", values="sdr_imp")

    print("\n[2] 対にした比較（同じ部屋どうしの差、部屋 10 通り）。p はウィルコクソン、4 つをホルム法で補正")
    tests, res = [], []
    for a_, b_ in (("Net", "ILRMA"), ("IRM（oracle）", "Net")):
        for r in sorted(wide.rt60.unique()):
            d = (piv.loc[r, a_] - piv.loc[r, b_]).to_numpy()
            res.append((a_, b_, r) + paired(d))
    adj = holm([x[5] for x in res])
    for (a_, b_, r, mu, ci, p, npos), pa in zip(res, adj):
        print(f"  {a_} − {b_}、RT60 {r:.2f} s：平均 {mu:+6.2f} dB、95% t 区間 [{ci[0]:+6.2f}, {ci[1]:+6.2f}]、"
              f"正の部屋 {npos}/{len(piv.loc[r])}、p = {p:.4f}、補正後 {pa:.4f}")

    print("\n[3] 線形混合モデル：sdr_imp ~ 方法 × 残響 ＋（部屋の変量効果）。基準は ILRMA・RT60 0.15 s")
    d3 = wide[wide.method.isin(["Net", "ILRMA", "IRM（oracle）"])].copy()
    d3["m"] = d3.method.map({"Net": "Net", "ILRMA": "ILRMA", "IRM（oracle）": "IRM"})
    d3["rt"] = d3.rt60.map(lambda r: f"r{int(round(r * 1000))}")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit = smf.mixedlm("sdr_imp ~ C(m, Treatment('ILRMA')) * C(rt)", d3, groups=d3["seed"]).fit(reml=True)
    ci = fit.conf_int()
    for k in fit.fe_params.index:
        nm = k.replace("C(m, Treatment('ILRMA'))[T.", "方法=").replace("C(rt)[T.", "残響=").replace("]", "")
        print(f"  {nm:28s} {fit.fe_params[k]:+7.2f}  [{ci.loc[k, 0]:+7.2f}, {ci.loc[k, 1]:+7.2f}]")
    print(f"  部屋の変量効果の標準偏差 {np.sqrt(fit.cov_re.iloc[0, 0]):.2f} dB、残差の標準偏差 {np.sqrt(fit.scale):.2f} dB")

    print("\n[4] ネットワークのばらつきの内訳（best.pt、irm の 3 つの種 × 部屋 10 × 残響 2）")
    b = net[(net.ckpt == "best") & (net.tag == "irm")].copy()
    b["rt"] = b.rt60.map(lambda r: f"r{int(round(r * 1000))}")
    b["one"] = 1
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        f4 = smf.mixedlm("sdr_imp ~ C(rt)", b, groups=b["one"], re_formula="0",
                         vc_formula={"room": "0 + C(seed)", "tseed": "0 + C(tseed)"}).fit(reml=True)
    vc = dict(zip(f4.model.exog_vc.names, np.sqrt(f4.vcomp)))
    print(f"  部屋による標準偏差 {vc['room']:.2f} dB、学習の種による標準偏差 {vc['tseed']:.2f} dB、残り {np.sqrt(f4.scale):.2f} dB")
    for t, g in b.groupby("tseed"):
        print(f"  学習の種 {t}：" + "、".join(f"RT60 {r:.2f} s {h.sdr_imp.mean():5.2f}" for r, h in g.groupby("rt60"))
              + f"（best は {int(g.step.iloc[0])} 回）")

    print("\n[5] 統計の誤用")
    print(" (a) 学習の種 1 つだけで Net − ILRMA を比べると（部屋 10 通りの平均の差）")
    for t, g in b.groupby("tseed"):
        cells = []
        for r, h in g.groupby("rt60"):
            d = h.set_index("seed").sdr_imp - piv.loc[r, "ILRMA"]
            cells.append(f"RT60 {r:.2f} s {d.mean():+5.2f} dB（p = {stats.wilcoxon(d).pvalue:.3f}）")
        print(f"  種 {t}：" + "、".join(cells))
    allck = net[(net.tag == "irm") & (net.ckpt != "best")]
    if len(allck):
        print(" (b) 途中の回の選び方（学習の種ごと。テストの部屋 20 通りの SDR 改善量の平均）")
        for t, g in allck.groupby("tseed"):
            m = g.groupby("step").sdr_imp.mean()
            vstep = int(b[b.tseed == t].step.iloc[0])
            print(f"  種 {t}：テストで一番よい回 {m.idxmax():5d} 回 → {m.max():.2f} dB【誤り】、"
                  f"検証で選んだ回 {vstep:5d} 回 → {m.loc[vstep]:.2f} dB【正しい】、最後の回 {m.index.max()} 回 → {m.iloc[-1]:.2f} dB")
    lk = net[(net.ckpt == "best") & (net.tag == "leak")]
    if len(lk):
        print(" (c) データ漏れ（テストの区間も学習に入れた）。同じ学習の種・同じ部屋どうしの差 漏れあり − 漏れなし")
        m = lk.merge(b, on=["tseed", "rt60", "seed"], suffixes=("_lk", ""))
        for r, h in m.groupby("rt60"):
            d = h.sdr_imp_lk - h.sdr_imp
            print(f"  RT60 {r:.2f} s：{d.mean():+.2f} dB（漏れありの平均 {h.sdr_imp_lk.mean():.2f}、なし {h.sdr_imp.mean():.2f}）")
    print(" (d) 平均だけで比べる：ILRMA（2ch）と PSM（oracle、1ch）の部屋ごとの勝ち負け（ILRMA − PSM）")
    for r in sorted(wide.rt60.unique()):
        d = piv.loc[r, "ILRMA"] - piv.loc[r, "PSM（oracle）"]
        print(f"  RT60 {r:.2f} s：ILRMA が勝った部屋 {int((d > 0).sum())}/10、差の範囲 {d.min():+.2f}〜{d.max():+.2f} dB、平均 {d.mean():+.2f} dB")
    both = piv["ILRMA"] - piv["PSM（oracle）"]
    print(f"  2 条件をまとめた平均の差 {both.mean():+.2f} dB（条件で向きが逆なので、まとめた平均だけでは何も言えない）")

    ps = net[(net.ckpt == "best") & (net.tag == "psa")]
    if len(ps):
        print("\n[6] 損失の比較 PSA − IRM（best.pt）。学習の種ごとに、部屋 20 通りの平均の差")
        m = ps.merge(b, on=["tseed", "rt60", "seed"], suffixes=("_psa", ""))
        d_seed = m.groupby("tseed").apply(lambda g: (g.sdr_imp_psa - g.sdr_imp).mean(), include_groups=False)
        print("  種ごとの差：" + "、".join(f"種 {t} {v:+.2f} dB" for t, v in d_seed.items()))
        a_ = m.groupby("tseed").sdr_imp_psa.mean().to_numpy()
        b_ = m.groupby("tseed").sdr_imp.mean().to_numpy()
        se_pair = d_seed.std(ddof=1) / np.sqrt(len(d_seed))
        se_unpair = np.sqrt(a_.var(ddof=1) / len(a_) + b_.var(ddof=1) / len(b_))
        print(f"  差の平均 {d_seed.mean():+.2f} dB、種で対にした標準誤差 {se_pair:.3f}、対にしないときの標準誤差 {se_unpair:.3f}"
              f"（種が 3 つでは区間は広い：t(0.975, 2) = {stats.t.ppf(0.975, 2):.2f}）")
        for r, h in m.groupby("rt60"):
            dd = (h.groupby("seed").sdr_imp_psa.mean() - h.groupby("seed").sdr_imp.mean())
            mu, ci, p, npos = paired(dd)
            print(f"  RT60 {r:.2f} s（部屋を単位、種 3 つの平均どうし）：{mu:+.2f} dB [{ci[0]:+.2f}, {ci[1]:+.2f}]、正の部屋 {npos}/{len(dd)}")


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
row = lambda m: [float(v) for v in re.search(rf"(?m)^  {re.escape(m)}\s+RT60 0\.15 s：\s*([-\d.]+) ± [\d.]+\s+RT60 0\.30 s：\s*([-\d.]+) ±", cap.stdout).groups()]
for m, v in (("ILRMA", (16.76, 5.66)), ("IRM（oracle）", (12.21, 11.93)), ("PSM（oracle）", (14.47, 14.24)), ("oracleW", (20.32, 10.91))):
    got = row(m)
    ok(all(abs(a - b) < 0.1 for a, b in zip(got, v)), f"[1] {m} が {v[0]}・{v[1]} dB（{got[0]}・{got[1]}。学習を使わないので教科書と同じ）")
net = row("Net")
ok(all(n < i for n, i in zip(net, row("IRM（oracle）"))), f"[1] Net（{net}）は理想比マスクに届かない（本番では 3.06・2.59 dB）")
d = {(a, r): num(rf"{re.escape(a)}、RT60 {r} s：平均\s+([-+\d.]+) dB") for a in ("Net − ILRMA", "IRM（oracle） − Net") for r in ("0.15", "0.30")}
ok(d[("Net − ILRMA", "0.15")] < 0 and d[("Net − ILRMA", "0.30")] < 0 and d[("Net − ILRMA", "0.30")] > d[("Net − ILRMA", "0.15")],
   f"[2] Net − ILRMA がどちらの残響でも負で、0.3 s では差が縮む {d[('Net − ILRMA', '0.15')]}・{d[('Net − ILRMA', '0.30')]}（本番では −13.69・−3.07 dB）")
ok(d[("IRM（oracle） − Net", "0.15")] > 0 and d[("IRM（oracle） − Net", "0.30")] > 0, "[2] IRM（oracle）− Net がどちらの残響でも正")
ok(abs(num(r"Intercept\s+([-+\d.]+)") - 16.76) < 0.1, "[3] 線形混合モデルの切片（ILRMA・0.15 s）が 16.76 dB")
sd = [num(r"部屋による標準偏差 ([\d.]+) dB"), num(r"学習の種による標準偏差 ([\d.]+) dB"), num(r"学習の種による標準偏差 [\d.]+ dB、残り ([\d.]+) dB")]
print(f"[4] 部屋による標準偏差 {sd[0]}、学習の種による {sd[1]}、残り {sd[2]} dB（本番では 0.15・0.46・0.21 dB。学習の種のほうが大きい）")
for m in re.finditer(r"種 (\d)：テストで一番よい回\s+\d+ 回 → ([\d.-]+) dB【誤り】、検証で選んだ回\s+\d+ 回 → ([\d.-]+) dB【正しい】", cap.stdout):
    ok(float(m.group(2)) >= float(m.group(3)), f"[5] (b) 種 {m.group(1)}：テストで選んだ回（{m.group(2)}）は検証で選んだ回（{m.group(3)}）以上（楽観的なかたより）")
ok("ILRMA が勝った部屋 7/10" in cap.stdout and "ILRMA が勝った部屋 0/10" in cap.stdout and abs(num(r"0\.15 s：ILRMA が勝った部屋.*?平均 ([-+\d.]+) dB") - 2.29) < 0.1
   and abs(num(r"0\.30 s：ILRMA が勝った部屋.*?平均 ([-+\d.]+) dB") + 8.58) < 0.1 and abs(num(r"まとめた平均の差 ([-+\d.]+) dB") + 3.14) < 0.1,
   "[5] (d) ILRMA − PSM の符号が残響の条件で逆（0.15 s で 7/10・+2.29 dB、0.3 s で 0/10・−8.58 dB、まとめると −3.14 dB）")
ok("t(0.975, 2) = 4.30" in cap.stdout, "[6] PSA − IRM を学習の種で対にした（種ごとの差は試しの設定では本番と違う。値は 3-12）")

## 6. 確認問題の答え合わせ（手順 H、`quiz_check.py`）

確認問題（教科書の 7 節）は、**先に紙で解いてから**実行してください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-7：確認問題の数値の答え合わせ（common/metrics.py と masks.py を使う）

使い方: cd ~/home-lab/partIV/iv7 && uv run python quiz_check.py
"""
import itertools
import pathlib
import sys

import numpy as np
from scipy import stats

HERE = pathlib.Path(".").resolve()   # ノートブック版：このノートブックのフォルダ（out/ と runs/ もここ）
sys.path.insert(0, ".")   # ノートブック版：common/・bss.py・bss6.py・masks.py・masknet.py はこのノートブックの隣
from common import metrics as M                         # noqa: E402
import masks                                            # noqa: E402

s = np.array([3.0, 0.0, 4.0])
e = np.array([2.0, 2.0, 2.0])
a = e @ s / (s @ s)
print(f"問1：α = {a:.4f}、‖αs‖² = {np.sum((a * s) ** 2):.4f}、‖ŝ − αs‖² = {np.sum((e - a * s) ** 2):.4f}、"
      f"SI-SDR = {M.si_sdr(s, e):.3f} dB、SNR = {M.snr(s, e):.3f} dB")
rho2 = (e @ s) ** 2 / ((s @ s) * (e @ e))
print(f"問2：3 倍の推定で SI-SDR = {M.si_sdr(s, 3 * e):.3f} dB、SNR = {M.snr(s, 3 * e):.3f} dB、"
      f"ρ² = {rho2:.4f}、SNR の最大 −10log10(1−ρ²) = {-10 * np.log10(1 - rho2):.3f} dB（c = {e @ s / (e @ e):.4f} のとき "
      f"{M.snr(s, (e @ s / (e @ e)) * e):.3f} dB）")
for n in (1600, 16000):
    L = 512
    q = L / n * (1 - (L - 1) / (2 * n))
    print(f"問3：n = {n}、L = {L}：q = {q:.4f}、SDR の近似 {10 * np.log10((1 + q) / (1 - q)):.2f} dB")
v = np.array([20.0, 5.0, -10.0])
print(f"問4：dB の平均 {v.mean():.2f}、比の平均の dB {10 * np.log10(np.mean(10 ** (v / 10))):.2f}、中央値 {np.median(v):.2f}")
S = np.array([[[3.0 + 0j]], [[-1.0 + 2j]]])               # (N=2, F=1, T=1)
X = S.sum(axis=0)
Mk = masks.ideal_masks(S, X)
for nm, m in Mk.items():
    err = abs(S[0, 0, 0] - m[0, 0, 0] * X[0, 0]) ** 2
    print(f"問5：{nm}：m = {np.round(m[0, 0, 0], 4)}、|s − m x|² = {err:.4f}")
print(f"     切らない複素マスク s/x = {S[0, 0, 0] / X[0, 0]:.4f}（大きさ {abs(S[0, 0, 0] / X[0, 0]):.4f}）")
C = np.array([[0.6, 0.1], [0.2, 0.5]])
for p in itertools.permutations(range(2)):
    print(f"問6：対応（正解 j ← 出力 p[j]）{p}：{np.mean([C[p[j], j] for j in range(2)]):.2f}")
print(f"     N = 3, 4, 5 の並べ方の数：{[len(list(itertools.permutations(range(k)))) for k in (3, 4, 5)]}")
d_mean, d_sd, R = 1.2, 1.5, 10
h = stats.t.ppf(0.975, R - 1) * d_sd / np.sqrt(R)
print(f"問7：t(0.975, 9) = {stats.t.ppf(0.975, 9):.3f}、半幅 {h:.3f}、区間 [{d_mean - h:.2f}, {d_mean + h:.2f}]")
p = np.array([0.004, 0.030, 0.020, 0.200])
order = np.argsort(p)
adj = np.empty_like(p)
run = 0.0
for i, j in enumerate(order):
    run = max(run, (len(p) - i) * p[j])
    adj[j] = min(1.0, run)
print(f"問8：ホルム法の補正後 {adj}（0.05 未満は {np.sum(adj < 0.05)} つ）、ウィルコクソン（n = 10）の p の最小 {2 / 2 ** 10:.4f}")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」）
ok(num(r"問1：.*SI-SDR = ([-\d.]+) dB") == 2.752 and num(r"問1：.*SNR = ([-\d.]+) dB") == 4.437, "問1 SI-SDR 2.752 dB、SNR 4.437 dB")
ok(num(r"問2：3 倍の推定で SI-SDR = ([-\d.]+) dB") == 2.752 and num(r"SNR の最大 .*? = ([\d.]+) dB") == 4.601, "問2 3 倍でも SI-SDR 2.752 dB、SNR の最大 4.601 dB")
ok(num(r"n = 1600、L = 512：.*?近似 ([\d.]+) dB") == 2.39 and num(r"n = 16000、L = 512：.*?近似 ([\d.]+) dB") == 0.27, "問3 2.39 dB と 0.27 dB")
ok(num(r"問4：dB の平均 ([\d.]+)") == 5.0 and num(r"比の平均の dB ([\d.]+)") == 15.37, "問4 5.00 と 15.37")
ok(num(r"PSM：m = ([\d.]+)") == 0.75 and num(r"PSM：m = [\d.]+、\|s − m x\|² = ([\d.]+)") == 4.5, "問5 PSM 0.75・誤差 4.50")
ok("(0, 1)：0.55" in cap.stdout and "(1, 0)：0.15" in cap.stdout and "[6, 24, 120]" in cap.stdout, "問6 0.55 と 0.15（入れ替えた対応を選ぶ）、6・24・120 通り")
ok("区間 [0.13, 2.27]" in cap.stdout, "問7 [0.13, 2.27]")
ok("[0.016 0.06  0.06  0.2  ]（0.05 未満は 1 つ）" in cap.stdout and "p の最小 0.0020" in cap.stdout, "問8 補正後 0.016・0.06・0.06・0.2（1 つ）、p の最小 0.0020")

## おわりに

- 評価指標（SDR・SI-SDR）の実装と照合、評価の落とし穴（尺度・フィルタ長・参照信号・短い区間・平均のとり方・改善量の基準）、理想マスクの上限と ILRMA の比較、PIT で学習する小さなマスク推定ネットワーク、部屋と学習の種を単位にした統計と、統計の誤用を、数値と図と音で確かめました。
- 4 節の学習を教科書の本番（2000 回、250 回ごと）で行うと、ネットワークの SDR 改善量・検証の SI-SDR の頭打ち・[4]〜[6] の値が教科書と比べられます。本番にするときは、4 節の 2 つのセルの `--steps`・`--eval-every` を消し、4 節の最初のセルから 5 節までを実行し直してください。
- このノートブックに入れていないもの：論文を読む手順 I（Le Roux ら（2019）の SI-SDR、Kolbæk ら（2017）の uPIT を、3-3・3-9 の式と対応させる）。